# LeetCode 431 — Encode N-ary Tree to Binary Tree bằng Python (Hard, Premium)

> **Lưu ý:** đây là bài **Premium** (bị khóa trên LeetCode, cần tài khoản trả phí mới mở được). Phần đề, ví dụ, constraints và khung code dưới đây được ghi lại theo bản LeetCode; notebook tự định nghĩa lại `Node`, `TreeNode` và mô phỏng cách LeetCode chấm để bạn chạy được mọi thứ ngay trong notebook.

Khung code Python3 mà LeetCode cho sẵn:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val: Optional[int] = None, children: Optional[List['Node']] = None):
        self.val = val
        self.children = children
"""

"""
# Definition for a binary tree node.
class TreeNode:
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None
"""

class Codec:
    # Encodes an n-ary tree to a binary tree.
    def encode(self, root: 'Optional[Node]') -> Optional[TreeNode]:


    # Decodes your binary tree to an n-ary tree.
    def decode(self, data: Optional[TreeNode]) -> 'Optional[Node]':


# Your Codec object will be instantiated and called as such:
# codec = Codec()
# codec.decode(codec.encode(root))
```

Mục tiêu của bài học:

1. Hiểu **cây N-ary** (mỗi node có bao nhiêu con cũng được) và **cây nhị phân** (mỗi node tối đa 2 con) từ con số 0; tự dựng cây N-ary từ định dạng level-order của LeetCode `[1,null,3,2,4,null,5,6]` và **in cả hai loại cây dạng ASCII** để nhìn thấy chúng.
2. Hiểu chỗ "khó" thật sự của đề: một node có thể có 3, 10, 10000 con nhưng node nhị phân chỉ có **2 ô** `left` / `right` — làm sao nhét vừa mà **không mất thông tin**?
3. Đi qua các cách theo thứ tự tự nhiên: cách **"lười"** (serialize thành dãy số rồi xếp thành một chuỗi node) → **left-child right-sibling** (con đầu → `left`, em kế → `right`; cách chính, vẽ hình, đệ quy, trace kỹ) → **biến thể** đảo vai `left`/`right` → **bản BFS lặp** bằng queue cho cả encode và decode.
4. Nắm các công cụ Python cần cho bài: class và tham chiếu, `deque`, đệ quy trên cây N-ary, `iter`/`next`, giới hạn đệ quy.
5. Chứng minh vì sao cách chính đúng (một **song ánh** giữa cây N-ary và cây nhị phân), phân tích độ phức tạp, test round-trip trên cây N-ary ngẫu nhiên có `seed` bằng hàm so sánh cấu trúc `same_nary`, tránh các lỗi kinh điển (đệ quy theo anh em làm tràn stack, lưu trạng thái trong `self`, `children = None`...) và nhận dạng pattern cho các bài anh em (428, 297, 589, 590, 559...).

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Hãy thiết kế một thuật toán để **mã hóa** (encode) một **cây N-ary** thành một **cây nhị phân**, và **giải mã** (decode) cây nhị phân đó để lấy lại cây N-ary ban đầu.
>
> Cây N-ary là cây có gốc (rooted tree) trong đó mỗi node có **không quá N con**. Tương tự, cây nhị phân là cây có gốc trong đó mỗi node có **không quá 2 con**.
>
> Không có giới hạn nào về cách thuật toán encode/decode của bạn hoạt động. Bạn chỉ cần bảo đảm một cây N-ary có thể được encode thành một cây nhị phân, và cây nhị phân này có thể được decode trở lại **đúng cấu trúc cây N-ary ban đầu**.
>
> *Input của cây N-ary được serialize theo thứ tự level-order (duyệt theo tầng), mỗi nhóm con được ngăn cách bởi giá trị `null` (xem ví dụ).*
>
> Đề có kèm một hình minh họa một cách encode cây 3-ary `[1,null,3,2,4,null,5,6]` thành cây nhị phân, và ghi chú: *hình đó chỉ là một ví dụ, có thể chạy được hoặc không. Bạn không nhất thiết phải theo định dạng đó, hãy sáng tạo và tự nghĩ ra những cách tiếp cận khác.*

**Example 1**

```text
Input: root = [1,null,3,2,4,null,5,6]
Output: [1,null,3,2,4,null,5,6]
```

**Example 2**

```text
Input: root = [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]
Output: [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]
```

**Example 3**

```text
Input: root = []
Output: []
```

**Constraints**

- Số node trong cây nằm trong khoảng `[0, 10^4]`.
- `0 <= Node.val <= 10^4`
- Chiều cao của cây N-ary **không vượt quá** `1000`.
- Không được dùng biến thành viên của class / biến global / biến static để lưu trạng thái. Thuật toán encode và decode phải **stateless** (không trạng thái).

### Ký hiệu dùng suốt notebook

- `n` = số node của cây N-ary (`0 <= n <= 10^4`, nghĩa là cây **có thể rỗng**).
- `H` = chiều cao cây N-ary (số node trên đường dài nhất từ root xuống một lá), `H <= 1000`.
- `W` = độ rộng lớn nhất của một tầng (số node nhiều nhất trên cùng một tầng).
- Để dễ đọc, khi nói về cây nhị phân đã encode, ta gọi node nhị phân có giá trị `x` là `B(x)`, còn node N-ary là `N(x)`.

### 1.1 Đề thực sự kiểm tra điều gì?

Đọc kỹ 2 dòng comment cuối khung code: LeetCode tạo **một** object `codec`, gọi `codec.encode(root)` để được một **cây nhị phân** (không phải chuỗi!), rồi đưa cây nhị phân đó cho `codec.decode(...)`, cuối cùng in cây N-ary kết quả theo định dạng list và so với input. Vì vậy Output luôn trùng Input: "cây bạn decode ra phải giống hệt cây ban đầu".

| Chi tiết của đề | Hệ quả khi thiết kế |
|---|---|
| `encode` phải trả về **`TreeNode`** (hoặc `None`) | không được trả chuỗi; mọi thông tin phải nằm trong các node nhị phân (`val`, `left`, `right`) |
| yêu cầu **stateless** | không được "giấu" cây gốc trong `self` hay biến global rồi `decode` trả lại; cây nhị phân phải **tự chứa đủ thông tin** |
| định dạng do ta **tự chọn** | được sáng tạo — miễn là `decode(encode(T))` cho lại đúng cây `T` |
| **thứ tự các con** quan trọng | `[1,null,3,2,4]` và `[1,null,2,3,4]` là hai cây **khác nhau**; decode phải giữ đúng thứ tự `3, 2, 4` |
| `n` có thể bằng `0` | `encode(None)` phải trả một giá trị hợp lệ (tự nhiên nhất: `None`) và `decode(None)` phải trả `None` |
| `n` tới `10^4`, một node có thể có tới `10^4 - 1` con | cây "hình sao" (root có 9999 con) là input hợp lệ — đệ quy theo **anh em** sẽ sâu 9999 tầng và vỡ (mục 11) |
| chiều cao `H <= 1000` | đệ quy theo **chiều sâu** chỉ sâu tối đa khoảng 1000 tầng — sát giới hạn mặc định của Python (`1000`), nên cần để ý (mục 11) |
| `0 <= val <= 10^4` | giá trị **có thể bằng 0** và có thể **trùng nhau** — không được dùng `if node.val:` hay dùng giá trị để nhận diện node |
| `Node.children` có thể là `None` | constructor `Node(val)` để `children = None`; code phải chịu được cả `None` lẫn `[]` |

Mỗi dòng của bảng ứng với một lỗi kinh điển ở mục 14.

## 2. Nền tảng: cây N-ary từ căn bản

### 2.1 Từ vựng

**Cây N-ary** (đọc là "en-a-ri", nghĩa là "N nhánh") gồm các **node**. Mỗi node có hai thứ:

- `val`: giá trị lưu ở node.
- `children`: một **list** các node con, **có thứ tự**: con đầu tiên, con thứ hai, ... List này có thể rỗng (node lá).

Cây của Example 1, `[1,null,3,2,4,null,5,6]`, trông như sau:

```text
        1
     /  |  \
    3   2   4
   / \
  5   6
```

| Thuật ngữ | Nghĩa | Ví dụ trên cây Example 1 |
|---|---|---|
| **root** (gốc) | node trên cùng, không có cha | `1` |
| **children** (các con) | list node ngay dưới, theo thứ tự trái → phải | `children(1) = [3, 2, 4]` |
| **first child** (con đầu, "con cả") | phần tử đầu của `children` | con đầu của `1` là `3` |
| **siblings** (anh em) | các node có chung cha | `3, 2, 4` là anh em |
| **next sibling** (em kế) | node đứng ngay sau trong list của cha | em kế của `3` là `2`, của `2` là `4`, của `4` thì **không có** |
| **leaf** (lá) | node không có con | `5, 6, 2, 4` |
| **height** (chiều cao) | số node trên đường dài nhất từ root xuống lá | `3` (đường `1 → 3 → 5`) |

Khác cây nhị phân, ở đây **không có** khái niệm "con trái / con phải" — chỉ có "con thứ 1, con thứ 2, ...". Ta định nghĩa `Node` **y hệt** LeetCode và dựng tay cây Example 1.

Một điểm Python quan trọng: `self.children` chứa **tham chiếu** (reference) tới các object `Node` khác — hãy hình dung đó là "địa chỉ" của node con. Dựng cây chính là tạo các node rồi nối tham chiếu.

In [1]:
from collections import deque
from typing import List, Optional
import random
import sys


class Node:
    def __init__(self, val: Optional[int] = None, children: Optional[List['Node']] = None):
        self.val = val
        self.children = children


# Dựng TAY cây Example 1: tạo các lá trước, rồi tạo cha với list con.
n5, n6 = Node(5, []), Node(6, [])
n3 = Node(3, [n5, n6])
n2, n4 = Node(2, []), Node(4, [])
nary1 = Node(1, [n3, n2, n4])

print("root:", nary1.val)
print("các con của root:", [c.val for c in nary1.children])      # [3, 2, 4] - có thứ tự!
print("con đầu của 3   :", nary1.children[0].children[0].val)    # 5
print("số con của 2    :", len(n2.children))                      # 0 -> lá
print("Node(7).children:", Node(7).children)                      # None: constructor mặc định KHÔNG tạo list

root: 1
các con của root: [3, 2, 4]
con đầu của 3   : 5
số con của 2    : 0
Node(7).children: None


### 2.2 Định dạng level-order của cây N-ary và hàng đợi `deque`

LeetCode viết cây N-ary thành list **theo tầng**, và dùng `null` để **kết thúc một nhóm con**. Cách đọc `[1,null,3,2,4,null,5,6]`:

```text
1           <- root
null        <- kết thúc "nhóm" của tầng root (luôn có ngay sau root)
3, 2, 4     <- nhóm con của node 1 (node thật đầu tiên)
null        <- hết con của 1
5, 6        <- nhóm con của node 3 (node thật thứ hai)
            <- (các null cuối bị lược bỏ: con của 3 xong, 2 và 4 không có con)
```

Quy tắc: lấy lần lượt từng node thật **theo thứ tự tầng** (1, rồi 3, 2, 4, rồi 5, 6...); mỗi node "tiêu thụ" một **nhóm** các giá trị liên tiếp cho tới `null` kế tiếp làm list con. Node không có con thì nhóm của nó **rỗng** — tức là gặp `null` ngay. Ví dụ trong Example 2, đoạn `5,null,null,6,7` nghĩa là: hết con của 1 (`null` thứ nhất), node 2 không có con (`null` thứ hai ngay lập tức), rồi `6,7` là con của node 3.

Để xử lý "node nào gặp trước thì được nhận con trước", ta dùng **hàng đợi** (queue): vào trước, ra trước (FIFO — First In, First Out), giống hàng người xếp mua vé. Python có sẵn `collections.deque`:

- `q.append(x)`: thêm `x` vào **cuối** hàng.
- `q.popleft()`: lấy phần tử ở **đầu** hàng — nhanh `O(1)`. (Còn `list.pop(0)` phải dời mọi phần tử còn lại lên một ô, tốn `O(n)`.)

In [2]:
q = deque()
q.append("A")
q.append("B")
q.append("C")
print("hàng đợi:", list(q))
print("ra trước:", q.popleft())   # A vào trước nên ra trước
print("ra tiếp :", q.popleft())   # B
print("còn lại :", list(q))       # ['C']
print("rỗng chưa?", not q)        # False: vẫn còn 'C'

hàng đợi: ['A', 'B', 'C']
ra trước: A
ra tiếp : B
còn lại : ['C']
rỗng chưa? False


### 2.3 Helper: `build_nary` (list → cây N-ary) và `nary_to_list` (cây → list)

Hai hàm tiện ích để tạo cây test và in kết quả theo đúng kiểu LeetCode:

- `build_nary(values)`: tạo root từ `values[0]`, **bỏ qua** `values[1]` (luôn là `null`), rồi bắt đầu đọc từ chỉ số `i = 2`. Mỗi lần `popleft` một node cha, đọc liên tiếp các giá trị khác `None` làm con của nó (mỗi con cũng được xếp vào cuối queue để tới lượt nhận con), gặp `None` thì nhảy qua và chuyển sang cha kế tiếp. Lưu ý kiểm tra `values[i] is not None` chứ **không** viết `if values[i]:` — vì giá trị `0` là hợp lệ trong bài này nhưng lại là "falsy" trong Python.
- `nary_to_list(root)`: làm ngược lại — ghi `root`, `None`, rồi với mỗi node theo thứ tự tầng: ghi các con của nó và một `None` kết thúc nhóm; cuối cùng cắt các `None` thừa ở đuôi.

Cả hai đều viết **bằng vòng lặp** nên dùng được cho cây rất sâu. Nếu `build_nary` rồi `nary_to_list` cho lại đúng list ban đầu trên cả 3 ví dụ của đề, ta tin được hai helper.

In [3]:
def build_nary(values):
    # Dựng cây N-ary từ list level-order kiểu LeetCode (None = kết thúc một nhóm con).
    if not values or values[0] is None:
        return None
    root = Node(values[0], [])
    queue = deque([root])
    i = 2                                    # values[1] luôn là None sau root
    while queue and i < len(values):
        parent = queue.popleft()
        while i < len(values) and values[i] is not None:
            child = Node(values[i], [])
            parent.children.append(child)
            queue.append(child)
            i += 1
        i += 1                               # nhảy qua None kết thúc nhóm con của parent
    return root


def nary_to_list(root):
    # Cây N-ary -> list level-order (đúng cách LeetCode in Output).
    if root is None:
        return []
    out = [root.val, None]
    queue = deque([root])
    while queue:
        node = queue.popleft()
        for child in node.children or []:    # chịu được cả children = None
            out.append(child.val)
            queue.append(child)
        out.append(None)                     # kết thúc nhóm con của node
    while out and out[-1] is None:
        out.pop()
    return out


ex1 = [1, None, 3, 2, 4, None, 5, 6]
ex2 = [1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14]
ex3 = []
print(nary_to_list(build_nary(ex1)))
print(nary_to_list(nary1))                   # cây dựng tay ở trên cũng ra đúng list này
for vals in [ex1, ex2, ex3, [0], [0, None, 0, 0, None, 0]]:
    assert nary_to_list(build_nary(vals)) == vals
print("build_nary / nary_to_list OK")

[1, None, 3, 2, 4, None, 5, 6]
[1, None, 3, 2, 4, None, 5, 6]
build_nary / nary_to_list OK


### 2.4 Helper: in cây N-ary dạng ASCII

Đọc list level-order rất khó hình dung, nhất là Example 2. Hàm `print_nary` in cây theo kiểu **cây thư mục**: mỗi node một dòng, con nằm thụt vào dưới cha, `├──` là "còn em phía sau", `└──` là "con cuối cùng". Thứ tự từ trên xuống chính là thứ tự các con.

Hàm này tự gọi lại chính nó cho từng con — đó là **đệ quy**, sẽ học kỹ ở mục 3. Bây giờ chỉ cần dùng (và chỉ dùng cho cây nhỏ).

In [4]:
def print_nary(root):
    if root is None:
        print("(cây rỗng)")
        return
    print(root.val)

    def walk(node, prefix):
        kids = node.children or []
        for k, child in enumerate(kids):
            last = k == len(kids) - 1
            print(prefix + ("└── " if last else "├── ") + str(child.val))
            walk(child, prefix + ("    " if last else "│   "))

    walk(root, "")


print("Example 1:")
print_nary(build_nary(ex1))
print("\nExample 2:")
print_nary(build_nary(ex2))
print("\nExample 3:")
print_nary(build_nary(ex3))

Example 1:
1
├── 3
│   ├── 5
│   └── 6
├── 2
└── 4

Example 2:
1
├── 2
├── 3
│   ├── 6
│   └── 7
│       └── 11
│           └── 14
├── 4
│   └── 8
│       └── 12
└── 5
    ├── 9
    │   └── 13
    └── 10

Example 3:
(cây rỗng)


### 2.5 Cây nhị phân: `TreeNode`, `build_tree`, `tree_to_list`, `print_tree`

**Cây nhị phân** (binary tree): mỗi node có `val` và đúng **hai ô** con: `left` (con trái) và `right` (con phải), mỗi ô là một node hoặc `None`. Bài 431 dùng `TreeNode` "kiểu cũ": constructor chỉ nhận **một** tham số `x`, còn `left`/`right` luôn khởi đầu bằng `None` và ta tự gán sau. (Gọi `TreeNode(1, None, None)` sẽ bị `TypeError`.)

Khác biệt cốt lõi so với cây N-ary: ở cây nhị phân, "chỉ có con trái" và "chỉ có con phải" là **hai cây khác nhau**. Ví dụ `[1,2]` (2 là con trái) khác `[1,null,2]` (2 là con phải). Đây chính là "thông tin thừa" mà ta sẽ tận dụng: mỗi node nhị phân có 2 ô **có tên riêng**, và ta có quyền quy định ý nghĩa cho từng ô.

Bộ helper cho cây nhị phân (định dạng level-order của LeetCode cho cây nhị phân: mỗi node thật "tiêu thụ" đúng **2** ô kế tiếp làm trái/phải, `null` là ô trống):

- `build_tree(values)` / `tree_to_list(root)`: list ↔ cây nhị phân.
- `print_tree(root)`: in ASCII, `L:` là con trái, `R:` là con phải, `·` nghĩa là ô đó trống (chỉ in khi ô kia có con, để thấy rõ con còn lại nằm bên nào — cực kỳ quan trọng trong bài này).

In [5]:
class TreeNode:
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None


def build_tree(values):
    # Dựng cây nhị phân từ list level-order kiểu LeetCode (None = ô trống).
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def print_tree(root):
    if root is None:
        print("(cây rỗng)")
        return
    print(root.val)

    def walk(node, prefix):
        if node.left is None and node.right is None:
            return
        for tag, child, last in (("L", node.left, False), ("R", node.right, True)):
            branch = "└── " if last else "├── "
            if child is None:
                print(prefix + branch + tag + ": ·")
            else:
                print(prefix + branch + f"{tag}: {child.val}")
                walk(child, prefix + ("    " if last else "│   "))

    walk(root, "")


for vals in [[1, 2, 3, None, None, 4, 5], [1, 2], [1, None, 2], []]:
    assert tree_to_list(build_tree(vals)) == vals
print("[1,2]      (2 là con TRÁI):")
print_tree(build_tree([1, 2]))
print("[1,null,2] (2 là con PHẢI):")
print_tree(build_tree([1, None, 2]))

[1,2]      (2 là con TRÁI):
1
├── L: 2
└── R: ·
[1,null,2] (2 là con PHẢI):
1
├── L: ·
└── R: 2


## 3. Nền tảng: đệ quy trên cây, so sánh cấu trúc, `iter` / `next`

### 3.1 Đệ quy (recursion) trên cây N-ary

**Đệ quy** là khi một hàm **gọi lại chính nó** trên một bài toán **nhỏ hơn**. Cây N-ary hợp với đệ quy vì: *cây = root + danh sách các cây con*, mà mỗi cây con cũng là cây.

Một hàm đệ quy luôn có 2 phần:

1. **Base case** (trường hợp cơ sở): đủ nhỏ để trả lời ngay. Với cây N-ary: `node is None` (cây rỗng), hoặc node lá (vòng lặp qua `children` không chạy lần nào).
2. **Recursive case**: gọi hàm cho **từng con** bằng một vòng `for`, rồi kết hợp kết quả với `node.val`.

```text
count(node)  = 1 + tổng count(con) với mọi con
height(node) = 1 + max(height(con)) với mọi con   (lá: 1)
```

Với Example 1: `count(3) = 1 + 1 + 1 = 3`, `count(2) = count(4) = 1`, nên `count(1) = 1 + 3 + 1 + 1 = 6`. Chiều cao: `height(3) = 2`, nên `height(1) = 1 + max(2, 1, 1) = 3`.

**Duyệt pre-order / post-order** trên cây N-ary (LeetCode 589 / 590): pre-order ghi node **trước** rồi duyệt lần lượt các con; post-order duyệt hết các con **rồi mới** ghi node. (Cây N-ary không có in-order vì không có "giữa trái và phải".) Ta sẽ gặp lại hai thứ tự này ở mục 15 với một sự thật bất ngờ.

**Call stack:** khi `count(1)` gọi `count(3)`, nó tạm dừng chờ; các lời gọi đang chờ xếp chồng lên nhau trong **call stack** (ngăn xếp lời gọi). Chiều cao chồng = độ sâu của node đang xét, tối đa `H`. Python giới hạn chồng này (mặc định khoảng `1000`), vượt quá thì báo `RecursionError`. Đề cho `H <= 1000` — sát nút! Ghi nhớ chuyện này cho mục 11.

In [6]:
def count_nary(node):
    if node is None:
        return 0
    total = 1
    for c in node.children or []:
        total += count_nary(c)
    return total


def height_nary(node):
    if node is None:
        return 0
    best = 0                                 # lá: không có con -> best = 0 -> chiều cao 1
    for c in node.children or []:
        best = max(best, height_nary(c))
    return 1 + best


def preorder_nary(root):
    out = []
    def go(node):
        out.append(node.val)              # node TRƯỚC
        for c in node.children or []:
            go(c)                         # rồi lần lượt từng con
    if root is not None:
        go(root)
    return out


def postorder_nary(root):
    out = []
    def go(node):
        for c in node.children or []:
            go(c)                         # các con TRƯỚC
        out.append(node.val)              # node SAU CÙNG
    if root is not None:
        go(root)
    return out


t1 = build_nary(ex1)
print("số node:", count_nary(t1), "| chiều cao:", height_nary(t1))
print("pre-order :", preorder_nary(t1))
print("post-order:", postorder_nary(t1))
print("Example 2 -> số node:", count_nary(build_nary(ex2)), "| chiều cao:", height_nary(build_nary(ex2)))
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

số node: 6 | chiều cao: 3
pre-order : [1, 3, 5, 6, 2, 4]
post-order: [5, 6, 3, 2, 4, 1]
Example 2 -> số node: 14 | chiều cao: 5
giới hạn đệ quy hiện tại: 1000


### 3.2 So sánh hai cây theo **cấu trúc**: `same_nary` và `same_tree`

Để kiểm tra decode đúng hay sai, ta phải so cây decode ra với cây gốc. Viết `a == b` với hai `Node` là **không được**: class không định nghĩa phép so sánh, nên `==` chỉ hỏi "có phải **cùng một object** không". Hai cây giống hệt nhau nhưng gồm các object khác nhau vẫn cho `False`.

`same_nary(p, q)` so **cấu trúc**: hai node giống nhau khi cùng `val`, cùng **số con**, và con thứ `k` của bên này giống con thứ `k` của bên kia, với mọi `k`. Hai chi tiết:

- `children = None` và `children = []` được coi là như nhau (đều là lá) — LeetCode cũng in hai trường hợp này giống hệt nhau.
- Viết bằng **stack** tự quản lý (một `list`: `append` đặt lên đỉnh, `pop()` lấy từ đỉnh — vào sau ra trước, LIFO) thay vì đệ quy, để so được cả cây sâu hàng nghìn tầng mà không đụng giới hạn đệ quy.

`same_tree(p, q)` làm điều tương tự cho cây nhị phân (LeetCode 100): so `val`, rồi so trái với trái, phải với phải.

Thêm hai hàm dựng cây đặc biệt (bằng vòng lặp) để test: `make_star(k)` — root có `k` con lá (cây "hình sao", rất **rộng**), và `make_path(d)` — đường thẳng sâu `d` tầng (rất **sâu**).

In [7]:
def same_nary(p, q):
    stack = [(p, q)]                         # các cặp node còn chờ so sánh
    while stack:
        a, b = stack.pop()
        if a is None and b is None:
            continue
        if a is None or b is None or a.val != b.val:
            return False
        ca, cb = a.children or [], b.children or []    # None và [] coi như nhau
        if len(ca) != len(cb):
            return False
        stack.extend(zip(ca, cb))            # con thứ k so với con thứ k
    return True


def same_tree(p, q):
    stack = [(p, q)]
    while stack:
        a, b = stack.pop()
        if a is None and b is None:
            continue
        if a is None or b is None or a.val != b.val:
            return False
        stack.append((a.left, b.left))
        stack.append((a.right, b.right))
    return True


def make_star(k):
    return Node(0, [Node(i % 10_001, []) for i in range(1, k + 1)])


def make_path(d):
    root = Node(0, [])
    cur = root
    for i in range(1, d):
        child = Node(i % 10_001, [])
        cur.children.append(child)
        cur = child
    return root


a, b = build_nary(ex1), build_nary(ex1)
print("a == b              :", a == b)                       # False: hai bộ object khác nhau
print("same_nary(a, b)     :", same_nary(a, b))              # True
print("đổi thứ tự con      :", same_nary(build_nary([1, None, 3, 2, 4]), build_nary([1, None, 2, 3, 4])))
print("None vs [] (lá)     :", same_nary(Node(7), Node(7, [])))
print("sao 9999 con, sâu   :", count_nary(make_star(9999)), "node,", "đường sâu 5000 so được:",
      same_nary(make_path(5000), make_path(5000)))

a == b              : False
same_nary(a, b)     : True
đổi thứ tự con      : False
None vs [] (lá)     : True
sao 9999 con, sâu   : 10000 node, đường sâu 5000 so được: True


### 3.3 `iter` và `next`: "băng chuyền" dùng chung

`iter(lst)` tạo một **iterator** — object **nhớ mình đang đọc tới đâu**. Mỗi lần `next(it)` trả về phần tử kế tiếp và tiến lên một bước; hết phần tử thì ném `StopIteration`. Hãy hình dung băng chuyền ở quầy thu ngân: ai lấy cũng lấy món **đầu tiên còn lại**, món đã lấy không quay lại.

Điểm quý: nhiều lời gọi hàm (kể cả lời gọi đệ quy ở nhiều độ sâu) cùng dùng **chung một iterator** thì cũng dùng chung vị trí đọc — không cần biến chỉ số truyền qua lại. Ta sẽ dùng nó ở cách 1 để đọc lại một dãy số.

In [8]:
it = iter([10, 20, 30])
print(next(it), next(it))                   # 10 20
print(next(it, "HẾT"), next(it, "HẾT"))     # 30 HẾT : next(it, mặc_định) không ném lỗi


def read_one(stream):
    return next(stream)


stream = iter("abcde")
print(read_one(stream), read_one(stream), read_one(stream))   # a b c: dùng chung vị trí đọc
print("còn lại trên băng chuyền:", list(stream))               # ['d', 'e']

10 20
30 HẾT
a b c
còn lại trên băng chuyền: ['d', 'e']


### 3.4 Cái khó thật sự: "N con" mà chỉ có "2 ô"

Thử nghĩ cách **trực giác nhất**: copy từng node N-ary thành một `TreeNode`, con thứ 1 → `left`, con thứ 2 → `right`. Ngay với Example 1, root `1` có **3** con `3, 2, 4`: con `4` không còn chỗ để đặt. Cây Example 2 thì root có **4** con. Và đề cho phép một node có tới gần `10^4` con.

Vậy "đặt con vào thẳng các ô" là không đủ. Có hai hướng thoát:

1. **Đừng cố giữ hình dạng** — biến cây N-ary thành một dãy số (serialize), rồi chỉ dùng cây nhị phân như một "cái hộp" để chứa dãy số đó (mục 4). Hợp lệ với bộ chấm, nhưng "ăn gian" tinh thần đề.
2. **Đổi ý nghĩa của 2 ô**: thay vì `left`/`right` = "con thứ 1 / con thứ 2", cho chúng ý nghĩa khác sao cho **mỗi node chỉ cần đúng 2 mối liên kết** dù có bao nhiêu con (mục 5–6). Đây là ý tưởng đẹp và là lời giải chính.

Cell dưới đếm nhanh: trên Example 1 và 2, bao nhiêu node có nhiều hơn 2 con (tức là cách "con 1 → left, con 2 → right" sẽ làm mất dữ liệu).

In [9]:
def nodes_with_more_than_2_children(root):
    bad, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        kids = node.children or []
        if len(kids) > 2:
            bad.append((node.val, len(kids)))
        stack.extend(kids)
    return bad


print("Example 1:", nodes_with_more_than_2_children(build_nary(ex1)))   # node 1 có 3 con
print("Example 2:", nodes_with_more_than_2_children(build_nary(ex2)))   # node 1 có 4 con

Example 1: [(1, 3)]
Example 2: [(1, 4)]


## 4. Cách 1 — Cách "lười": serialize thành dãy số, rồi xếp dãy số thành một chuỗi node

### 4.1 Bước 1: cây N-ary → dãy số

Ý tưởng của người mới thường là: "cây khó quá, biến nó thành **dãy số** trước đã". Một cách serialize đơn giản: duyệt **pre-order**, với mỗi node ghi **2 số**: giá trị của nó và **số con** của nó.

Example 1: node `1` có 3 con → `1, 3`; node `3` có 2 con → `3, 2`; lá `5` → `5, 0`; lá `6` → `6, 0`; lá `2` → `2, 0`; lá `4` → `4, 0`:

```text
[1, 3,  3, 2,  5, 0,  6, 0,  2, 0,  4, 0]
```

Đọc lại bằng **một băng chuyền** (`iter`) dùng chung: lấy 2 số `(val, k)`, tạo node, rồi gọi đệ quy **đúng `k` lần** để dựng `k` con theo thứ tự. Vì pre-order ghi node trước rồi mới tới toàn bộ cây con thứ nhất, cây con thứ hai..., thứ tự đọc khớp hoàn toàn với thứ tự ghi. Chỉ cần "số con" là đủ biết khi nào một nhóm con kết thúc — không cần ký hiệu `null`.

In [10]:
def nary_to_tokens(root):
    # pre-order: với mỗi node ghi (val, số con)
    tokens = []
    def go(node):
        kids = node.children or []
        tokens.append(node.val)
        tokens.append(len(kids))
        for c in kids:
            go(c)
    if root is not None:
        go(root)
    return tokens


def tokens_to_nary(tokens):
    if not tokens:
        return None
    stream = iter(tokens)
    def build():
        val, k = next(stream), next(stream)
        return Node(val, [build() for _ in range(k)])   # đúng k lần, theo thứ tự
    return build()


toks = nary_to_tokens(build_nary(ex1))
print(toks)
print(nary_to_list(tokens_to_nary(toks)))
assert nary_to_list(tokens_to_nary(nary_to_tokens(build_nary(ex2)))) == ex2

[1, 3, 3, 2, 5, 0, 6, 0, 2, 0, 4, 0]
[1, None, 3, 2, 4, None, 5, 6]


### 4.2 Bước 2: nhét dãy số vào "cây nhị phân" dạng chuỗi node

Đề đòi `encode` trả về `TreeNode`. Cách lười nhất: mỗi số trong dãy thành một `TreeNode`, nối chúng thành một **chuỗi** (chain) theo con `right` — y như một danh sách liên kết (linked list): `B(1) → B(3) → B(3) → B(2) → ...`. Ô `left` bỏ trống hết.

- `encode_lazy(root)`: `nary_to_tokens`, rồi dựng chuỗi node bằng vòng lặp (giữ con trỏ `tail` tới node cuối để nối thêm).
- `decode_lazy(b)`: đi dọc `right` để gom lại dãy số, rồi `tokens_to_nary`.

Cách này **qua được bộ chấm** (bộ chấm chỉ kiểm `decode(encode(root))`), và stateless (mọi thông tin nằm trong các `TreeNode`).

In [11]:
def encode_lazy(root):
    tokens = nary_to_tokens(root)
    dummy = TreeNode(-1)                 # node giả đứng đầu, để khỏi xét riêng node đầu tiên
    tail = dummy
    for t in tokens:
        tail.right = TreeNode(t)
        tail = tail.right
    return dummy.right                   # None nếu cây rỗng


def decode_lazy(b):
    tokens = []
    while b is not None:                 # đi dọc chuỗi right
        tokens.append(b.val)
        b = b.right
    return tokens_to_nary(tokens)


chain = encode_lazy(build_nary(ex1))
print("cây nhị phân (dạng list):", tree_to_list(chain))
print("vài tầng đầu:")
print_tree(build_tree(tree_to_list(chain)[:7]))     # chỉ in 4 node đầu của chuỗi cho gọn
for vals in [ex1, ex2, ex3]:
    assert nary_to_list(decode_lazy(encode_lazy(build_nary(vals)))) == vals
print("round-trip cách lười: OK trên 3 ví dụ")

cây nhị phân (dạng list): [1, None, 3, None, 3, None, 2, None, 5, None, 0, None, 6, None, 0, None, 2, None, 0, None, 4, None, 0]
vài tầng đầu:
1
├── L: ·
└── R: 3
    ├── L: ·
    └── R: 3
        ├── L: ·
        └── R: 2
round-trip cách lười: OK trên 3 ví dụ


### 4.3 Cách lười vướng ở đâu?

Nó **đúng** về mặt kỹ thuật, nhưng không đúng tinh thần bài và có nhược điểm thật:

1. **Cây nhị phân không phản ánh cấu trúc**: nó chỉ là một linked list chở dãy số. Muốn biết "node `3` có những con nào" phải đọc lại **từ đầu** chuỗi. Đề hỏi "encode cây thành cây", tức là một phép biến đổi **cấu trúc**; cách này thực chất là "serialize thành mảng" (bài 428) rồi đội lốt `TreeNode`.
2. **Tốn gấp đôi node**: `2n` node nhị phân cho `n` node N-ary (mỗi node tốn 2 ô số). Ta có thể ghi số con vào chung một `val` (ví dụ `val * 10^5 + k`), nhưng đó là "mã hóa số học" còn lười hơn, và sẽ vỡ nếu giá trị hay số con lớn hơn dự kiến.
3. **Chiều cao cây nhị phân = `2n`** (tới `2·10^4`): bất kỳ hàm đệ quy nào đi trên cây nhị phân này (in, so sánh, duyệt) đều sẽ tràn call stack.
4. **Vẫn cần đệ quy theo chiều sâu N-ary** ở `tokens_to_nary` (sâu `H`), nên chẳng được lợi gì so với cách chính.

Một cách **còn lười hơn** và **sai luật**: `encode` cất `root` vào `self.saved`, trả về đại một `TreeNode(0)`, rồi `decode` trả lại `self.saved`. Bộ chấm dùng chung một object nên có thể "qua", nhưng vi phạm thẳng yêu cầu stateless — demo ở mục 14.

Cell dưới đo số node và chiều cao cây nhị phân của cách lười, để đối chiếu với cách chính ở mục 6.

In [12]:
def count_bin(root):
    cnt, stack = 0, [root] if root else []
    while stack:
        node = stack.pop()
        cnt += 1
        stack.extend(c for c in (node.left, node.right) if c)
    return cnt


def height_bin(root):
    best, stack = 0, [(root, 1)] if root else []
    while stack:
        node, d = stack.pop()
        best = max(best, d)
        stack.extend((c, d + 1) for c in (node.left, node.right) if c)
    return best


for name, vals in [("Example 1", ex1), ("Example 2", ex2)]:
    t = build_nary(vals)
    e = encode_lazy(t)
    print(f"{name}: n = {count_nary(t)}, H = {height_nary(t)} | lazy: {count_bin(e)} node nhị phân, cao {height_bin(e)}")

Example 1: n = 6, H = 3 | lazy: 12 node nhị phân, cao 12
Example 2: n = 14, H = 5 | lazy: 28 node nhị phân, cao 28


## 5. Mấu chốt: mỗi node chỉ cần nhớ **con đầu** và **em kế**

### 5.1 Quan sát từ một cây gia phả

Hãy tưởng tượng một dòng họ rất đông con. Thay vì mỗi người cha phải nhớ **tên của tất cả** các con (có thể 10 người!), ta quy ước:

- Người cha chỉ nhớ **con cả** (con đầu tiên).
- Mỗi người con nhớ **em kế** của mình (người em sinh ngay sau).

Muốn liệt kê con của cha: hỏi cha "con cả là ai?" → hỏi con cả "em kế là ai?" → hỏi người đó "em kế là ai?"... cho tới khi không còn em. Danh sách con **có thứ tự** được khôi phục đủ, mà **mỗi người chỉ nhớ đúng 2 cái tên**: con cả và em kế.

"Đúng 2 cái tên" — trùng khớp với **2 ô** của node nhị phân! Đây là biểu diễn kinh điển **left-child right-sibling** (LCRS, "con trái — anh em phải"):

```text
B(x).left  = node nhị phân của CON ĐẦU của x       (đi XUỐNG một thế hệ)
B(x).right = node nhị phân của EM KẾ của x         (đi NGANG trong cùng nhóm anh em)
```

Danh sách con `[c1, c2, ..., ck]` của `x` biến thành: `x.left = c1`, `c1.right = c2`, `c2.right = c3`, ..., `ck.right = None`. Tức là danh sách con trở thành một **linked list** nối bằng `right`, và `left` của cha trỏ vào **đầu** linked list đó.

Cell dưới tính bảng "con đầu / em kế" của mọi node trên Example 1.

In [13]:
def first_child_next_sibling_table(root):
    rows = [(root.val, root.children[0].val if root.children else None, None)]   # root không có anh em
    queue = deque([root])
    while queue:
        node = queue.popleft()
        kids = node.children or []
        for k, child in enumerate(kids):
            first = child.children[0].val if child.children else None
            nxt = kids[k + 1].val if k + 1 < len(kids) else None
            rows.append((child.val, first, nxt))
            queue.append(child)
    return rows


print(f"{'node':>4} | {'con đầu':>7} | {'em kế':>5}")
for v, first, nxt in first_child_next_sibling_table(build_nary(ex1)):
    print(f"{v:>4} | {str(first):>7} | {str(nxt):>5}")

node | con đầu | em kế
   1 |       3 |  None
   3 |       5 |     2
   2 |    None |     4
   4 |    None |  None
   5 |    None |     6
   6 |    None |  None


### 5.2 Hình chuyển đổi Example 1

Ghép bảng trên lại: `left` lấy cột "con đầu", `right` lấy cột "em kế".

```text
   Cây N-ary                 Cùng các mũi tên, vẽ lại            Cây nhị phân thật sự
                             (↓ = left, → = right)               (xoay 45 độ)

        1                    1                                          1
     /  |  \                 ↓                                         /
    3   2   4                3 ──→ 2 ──→ 4                            3
   / \                       ↓                                       / \
  5   6                      5 ──→ 6                                5   2
                                                                     \   \
                                                                      6   4
```

Đọc hình giữa: mỗi **hàng ngang** là một nhóm anh em (nối bằng `right`), mỗi mũi tên **xuống** đi từ cha tới con cả (bằng `left`). Hình bên phải chỉ là hình giữa "xoay" cho đúng dạng cây nhị phân: `→` trở thành cạnh chéo sang **phải**, `↓` thành cạnh chéo sang **trái**.

Ba quan sát quan trọng (sẽ được chứng minh ở mục 13):

1. Số node nhị phân **bằng đúng** `n` — không tốn thêm node nào (cách lười tốn `2n`).
2. **Root nhị phân không bao giờ có con phải** (root N-ary không có anh em).
3. `B(x).left is None` ⇔ `x` là lá; `B(x).right is None` ⇔ `x` là con **cuối** của cha nó.

Dạng list của cây nhị phân bên phải là `[1,3,null,5,2,null,6,null,4]` — ta sẽ kiểm chứng bằng code ngay sau đây.

## 6. Cách 2 (cách chính) — Left-child right-sibling bằng đệ quy

### 6.1 Encode

`encode_lcrs(node)` trả về node nhị phân `B(node)` đã được nối đầy đủ phần bên dưới:

1. **Base case:** `node is None` → trả `None` (cây rỗng ↔ cây nhị phân rỗng).
2. Tạo `b = TreeNode(node.val)`.
3. Duyệt **lần lượt** các con `c1, c2, ..., ck`. Với mỗi con, gọi đệ quy `encode_lcrs(c)` để được `B(c)` (đã chứa cả cây con của `c`), rồi nối:
   - con **đầu tiên** → `b.left = B(c1)`;
   - con thứ `i >= 2` → `B(c_{i-1}).right = B(c_i)` (con trước trỏ `right` sang con này).
   Biến `prev` giữ node nhị phân của con vừa nối, để biết nối con kế tiếp vào đâu.
4. Trả `b`.

Chú ý: ta **duyệt anh em bằng vòng `for`**, chỉ **đệ quy xuống con**. Vì vậy độ sâu đệ quy = chiều cao N-ary `H`, không phải số anh em. (Mục 11 sẽ cho thấy cách viết "đệ quy cả sang em" làm vỡ chương trình với cây hình sao.)

In [14]:
def encode_lcrs(node):
    if node is None:
        return None
    b = TreeNode(node.val)
    prev = None                              # node nhị phân của con vừa nối
    for child in node.children or []:
        bc = encode_lcrs(child)              # B(child), đã chứa cả cây con của child
        if prev is None:
            b.left = bc                      # con đầu -> left của cha
        else:
            prev.right = bc                  # con sau -> right của anh ngay trước
        prev = bc
    return b


e1 = encode_lcrs(build_nary(ex1))
print("Example 1 sau encode:", tree_to_list(e1))
print_tree(e1)
assert tree_to_list(e1) == [1, 3, None, 5, 2, None, 6, None, 4]
print("encode(None) =", encode_lcrs(None))

Example 1 sau encode: [1, 3, None, 5, 2, None, 6, None, 4]
1
├── L: 3
│   ├── L: 5
│   │   ├── L: ·
│   │   └── R: 6
│   └── R: 2
│       ├── L: ·
│       └── R: 4
└── R: ·
encode(None) = None


### 6.2 Decode

Làm ngược lại: `decode_lcrs(b)` trả về node N-ary tương ứng với `b` (kèm toàn bộ con cháu):

1. **Base case:** `b is None` → `None`.
2. Tạo `node = Node(b.val, [])` — nhớ tạo **list rỗng** thật, đừng để `children = None` rồi `append` (lỗi 4, mục 14).
3. Các con của `node` chính là linked list bắt đầu ở `b.left` và nối bằng `right`. Dùng con trỏ `cur = b.left`; trong khi `cur` khác `None`: decode đệ quy `cur` thành một con, `append` vào `node.children`, rồi **đi sang em**: `cur = cur.right`.
4. Trả `node`.

Lại một lần nữa: anh em đi bằng **vòng `while`**, chỉ đệ quy **xuống** (`cur` là con của `node`), nên độ sâu đệ quy cũng là `H`.

Điều hay: decode **không cần biết trước** mỗi node có bao nhiêu con — linked list tự "kết thúc" khi gặp `right = None`. Thông tin "số con" mà cách lười phải ghi tường minh, ở đây được mã hóa **ngầm** trong vị trí của các `None`.

In [15]:
def decode_lcrs(b):
    if b is None:
        return None
    node = Node(b.val, [])
    cur = b.left                             # con đầu
    while cur is not None:
        node.children.append(decode_lcrs(cur))
        cur = cur.right                      # sang em kế
    return node


for name, vals in [("Example 1", ex1), ("Example 2", ex2), ("Example 3", ex3)]:
    original = build_nary(vals)
    back = decode_lcrs(encode_lcrs(original))
    assert same_nary(back, original) and nary_to_list(back) == vals
    print(name, "round-trip OK ->", nary_to_list(back))

Example 1 round-trip OK -> [1, None, 3, 2, 4, None, 5, 6]
Example 2 round-trip OK -> [1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14]
Example 3 round-trip OK -> []


### 6.3 Example 2 qua tay LCRS

Example 2 có root với 4 con. Hãy nhìn cây nhị phân của nó: dọc theo **cạnh phải** đi từ `2` là cả nhóm anh em `2 → 3 → 4 → 5`, mỗi node trong nhóm "thả" con cả của mình xuống bằng cạnh trái (`3 → 6`, `4 → 8`, `5 → 9`), còn `2` là lá nên ô `L` của nó trống. So sánh với hình N-ary ở mục 2.4.

In [16]:
e2 = encode_lcrs(build_nary(ex2))
print("số node nhị phân:", count_bin(e2), "| chiều cao nhị phân:", height_bin(e2))
print_tree(e2)

số node nhị phân: 14 | chiều cao nhị phân: 7
1
├── L: 2
│   ├── L: ·
│   └── R: 3
│       ├── L: 6
│       │   ├── L: ·
│       │   └── R: 7
│       │       ├── L: 11
│       │       │   ├── L: 14
│       │       │   └── R: ·
│       │       └── R: ·
│       └── R: 4
│           ├── L: 8
│           │   ├── L: 12
│           │   └── R: ·
│           └── R: 5
│               ├── L: 9
│               │   ├── L: 13
│               │   └── R: 10
│               └── R: ·
└── R: ·


## 7. Trace từng bước cách chính

### 7.1 Trace encode Example 1

Hàm `trace_encode` giống `encode_lcrs`, chỉ thêm `print` (thụt lề theo độ sâu đệ quy). Thứ tự sự kiện:

| Bước | Độ sâu | Sự kiện | Giải thích |
|---:|:---:|---|---|
| 1 | 0 | tạo `B(1)`, các con `[3, 2, 4]` | root |
| 2 | 1 | tạo `B(3)`, các con `[5, 6]` | vào con đầu của 1 |
| 3 | 2 | tạo `B(5)`, lá | con đầu của 3 |
| 4 | 1 | `B(3).left = B(5)` | 5 là con **đầu** của 3 → `left` |
| 5 | 2 | tạo `B(6)`, lá | con thứ 2 của 3 |
| 6 | 1 | `B(5).right = B(6)` | 6 là **em** của 5 → `right` của 5 |
| 7 | 0 | `B(1).left = B(3)` | encode(3) xong; 3 là con đầu của 1 |
| 8 | 1 | tạo `B(2)`, lá | con thứ 2 của 1 |
| 9 | 0 | `B(3).right = B(2)` | 2 là em của 3 |
| 10 | 1 | tạo `B(4)`, lá | con thứ 3 của 1 |
| 11 | 0 | `B(2).right = B(4)` | 4 là em của 2 |

Để ý bước 4 và 7: phép nối diễn ra **sau khi** lời gọi đệ quy của con trả về (vì ta cần `B(con)` đã dựng xong). Và bước 9: `B(2)` được nối vào `right` của `B(3)` — **không phải** của `B(1)`.

In [17]:
def trace_encode(node, depth=0):
    pad = "    " * depth
    b = TreeNode(node.val)
    kids = [c.val for c in node.children or []]
    print(f"{pad}tạo B({node.val})" + (f", các con {kids}" if kids else ", lá"))
    prev = None
    for child in node.children or []:
        bc = trace_encode(child, depth + 1)
        if prev is None:
            b.left = bc
            print(f"{pad}B({node.val}).left = B({child.val})    # con đầu")
        else:
            prev.right = bc
            print(f"{pad}B({prev.val}).right = B({child.val})   # em kế")
        prev = bc
    return b


tb = trace_encode(build_nary(ex1))
print("kết quả:", tree_to_list(tb))

tạo B(1), các con [3, 2, 4]
    tạo B(3), các con [5, 6]
        tạo B(5), lá
    B(3).left = B(5)    # con đầu
        tạo B(6), lá
    B(5).right = B(6)   # em kế
B(1).left = B(3)    # con đầu
    tạo B(2), lá
B(3).right = B(2)   # em kế
    tạo B(4), lá
B(2).right = B(4)   # em kế
kết quả: [1, 3, None, 5, 2, None, 6, None, 4]


### 7.2 Trace decode trên một cây "khó" hơn

Lấy cây N-ary `[1,null,2,3,4,null,5,null,null,6,7]`: root `1` có 3 con `2, 3, 4`; `2` có một con `5`; `3` là **lá kẹp giữa** hai anh có con; `4` có hai con `6, 7`. Cây nhị phân sau encode là `[1,2,null,5,3,null,null,null,4,6,null,null,7]`:

```text
   N-ary                    nhị phân (↓ left, → right)
      1                     1
    / | \                   ↓
   2  3  4                  2 ──→ 3 ──→ 4
   |    / \                 ↓           ↓
   5   6   7                5           6 ──→ 7
```

Decode đi như sau (`cur` là con trỏ chạy dọc linked list các con):

| Bước | Đang decode | `cur` | Hành động |
|---:|---|---|---|
| 1 | `B(1)` | `B(2)` = `B(1).left` | tạo `N(1)`; decode `B(2)` làm con thứ 1 |
| 2 | `B(2)` | `B(5)` | tạo `N(2)`; decode `B(5)` làm con thứ 1 của 2 |
| 3 | `B(5)` | `None` | tạo `N(5)`, `left` trống → lá, xong |
| 4 | `B(2)` | `B(5).right = None` | hết em → `N(2).children = [5]` |
| 5 | `B(1)` | `B(2).right = B(3)` | đi ngang: decode `B(3)` làm con thứ 2 của 1 |
| 6 | `B(3)` | `None` | lá → `N(3).children = []` |
| 7 | `B(1)` | `B(3).right = B(4)` | decode `B(4)` làm con thứ 3 |
| 8 | `B(4)` | `B(6)` rồi `B(7)` rồi `None` | `N(4).children = [6, 7]` |
| 9 | `B(1)` | `B(4).right = None` | hết em → `N(1).children = [2, 3, 4]`, xong |

In [18]:
def trace_decode(b, depth=0):
    pad = "    " * depth
    node = Node(b.val, [])
    print(f"{pad}tạo N({b.val}); con đầu = {'B(%d)' % b.left.val if b.left else 'None (lá)'}")
    cur = b.left
    while cur is not None:
        node.children.append(trace_decode(cur, depth + 1))
        nxt = cur.right
        print(f"{pad}N({b.val}).children = {[c.val for c in node.children]}; "
              f"em kế của {cur.val}: {'B(%d)' % nxt.val if nxt else 'None -> hết con'}")
        cur = nxt
    return node


hard = [1, None, 2, 3, 4, None, 5, None, None, 6, 7]
bh = encode_lcrs(build_nary(hard))
print("nhị phân:", tree_to_list(bh))
nh = trace_decode(bh)
print("N-ary   :", nary_to_list(nh))
assert nary_to_list(nh) == hard

nhị phân: [1, 2, None, 5, 3, None, None, None, 4, 6, None, None, 7]
tạo N(1); con đầu = B(2)
    tạo N(2); con đầu = B(5)
        tạo N(5); con đầu = None (lá)
    N(2).children = [5]; em kế của 5: None -> hết con
N(1).children = [2]; em kế của 2: B(3)
    tạo N(3); con đầu = None (lá)
N(1).children = [2, 3]; em kế của 3: B(4)
    tạo N(4); con đầu = B(6)
        tạo N(6); con đầu = None (lá)
    N(4).children = [6]; em kế của 6: B(7)
        tạo N(7); con đầu = None (lá)
    N(4).children = [6, 7]; em kế của 7: None -> hết con
N(1).children = [2, 3, 4]; em kế của 4: None -> hết con
N-ary   : [1, None, 2, 3, 4, None, 5, None, None, 6, 7]


## 8. Code nộp LeetCode (cách chính)

Chuyển hai hàm ở mục 6 vào class `Codec` với đúng tên và signature:

- `encode(self, root)` và `decode(self, data)` chỉ gọi hai **hàm lồng** (nested function) `enc` / `dec`. Không lưu gì vào `self` → **stateless** đúng yêu cầu.
- `root.children or []`: chịu được node có `children = None` (constructor mặc định của `Node`).
- Decode tạo `Node(data.val, [])` với list rỗng thật.
- Trên LeetCode, `Node`, `TreeNode`, `Optional`, `List` đã có sẵn; trong notebook ta đã định nghĩa ở mục 2.

Về giới hạn đệ quy: độ sâu đệ quy tối đa `H <= 1000`. Nếu sợ sát giới hạn, có thể thêm `sys.setrecursionlimit(10_000)` ở đầu file, hoặc dùng bản BFS ở mục 10 (không đệ quy).

In [19]:
class Codec:
    # Encodes an n-ary tree to a binary tree.
    def encode(self, root: 'Optional[Node]') -> Optional[TreeNode]:
        def enc(node):
            b = TreeNode(node.val)
            prev = None
            for child in node.children or []:
                bc = enc(child)
                if prev is None:
                    b.left = bc          # con đầu -> left
                else:
                    prev.right = bc      # em kế -> right của anh trước
                prev = bc
            return b

        return enc(root) if root is not None else None

    # Decodes your binary tree to an n-ary tree.
    def decode(self, data: Optional[TreeNode]) -> 'Optional[Node]':
        def dec(b):
            node = Node(b.val, [])
            cur = b.left
            while cur is not None:
                node.children.append(dec(cur))
                cur = cur.right
            return node

        return dec(data) if data is not None else None


codec = Codec()
for vals in [ex1, ex2, ex3, [0], [0, None, 0, 0, None, 0]]:
    out = nary_to_list(codec.decode(codec.encode(build_nary(vals))))
    assert out == vals, (vals, out)
print("Codec: 5 test PASS (3 ví dụ đề + 1 node + giá trị 0 trùng nhau)")

Codec: 5 test PASS (3 ví dụ đề + 1 node + giá trị 0 trùng nhau)


### 8.2 Mô phỏng đúng cách LeetCode chấm

Hàm `run_leetcode(codec_cls, values)` làm đúng những gì bộ chấm làm, cộng thêm vài kiểm tra "luật chơi":

1. Dựng cây N-ary từ `values` bằng `build_nary`.
2. `codec = codec_cls()`, `bin_root = codec.encode(root)`; kiểm tra kết quả là `TreeNode` hoặc `None` (không phải chuỗi, không phải `Node`).
3. **Kiểm tra stateless**: decode bằng một object **mới** `codec_cls()` — nếu codec "giấu" cây trong `self`, object mới không có gì để trả.
4. Trả về `nary_to_list(...)` của cây decode ra, để so với Output của đề.

Ngoài 3 ví dụ, test thêm các ca biên: 1 node, cây hình sao 9999 con, đường thẳng sâu 900 tầng (gần mức `H <= 1000`).

In [20]:
def run_leetcode(codec_cls, values):
    root = build_nary(values)
    bin_root = codec_cls().encode(root)
    assert bin_root is None or isinstance(bin_root, TreeNode), type(bin_root)
    ans = codec_cls().decode(bin_root)                 # object MỚI: phát hiện codec có trạng thái
    return nary_to_list(ans)


for name, vals in [("Example 1", ex1), ("Example 2", ex2), ("Example 3", ex3)]:
    out = run_leetcode(Codec, vals)
    assert out == vals
    print(f"{name}: Output = {out}")

for name, tree in [("1 node", Node(42)), ("sao 9999 con", make_star(9999)), ("đường sâu 900", make_path(900))]:
    vals = nary_to_list(tree)
    assert run_leetcode(Codec, vals) == vals
    print(f"{name}: PASS ({sum(v is not None for v in vals)} node)")

Example 1: Output = [1, None, 3, 2, 4, None, 5, 6]
Example 2: Output = [1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14]
Example 3: Output = []
1 node: PASS (1 node)
sao 9999 con: PASS (10000 node)
đường sâu 900: PASS (900 node)


## 9. Cách 3 — Biến thể: con đầu → `right`, em kế → `left`

Ở cách chính, quy ước "con đầu → `left`, em → `right`" là **tùy ý**. Đề nói "không có giới hạn nào về cách encode", nên ta hoàn toàn có thể **đổi vai**:

```text
B(x).right = B(con đầu của x)
B(x).left  = B(em kế của x)
```

Code giống hệt, chỉ đổi `left` ↔ `right` ở những chỗ nối và chỗ đọc (2 chỗ trong encode, 2 chỗ trong decode). Kết quả là **ảnh gương** (mirror) của cây nhị phân cách chính: lật trái ↔ phải ở mọi node. Ta kiểm chứng điều đó bằng hàm `mirror` (LeetCode 226 — Invert Binary Tree).

Tại sao đáng học biến thể này? Vì nó cho thấy điều cốt lõi **không phải** "left hay right", mà là: **một ô dành cho quan hệ cha → con cả, ô còn lại dành cho quan hệ anh → em**. Bất kỳ cách gán nhất quán nào cũng đúng. Còn một số biến thể khác cũng chạy được (ví dụ nối các con theo thứ tự **ngược**: cha trỏ tới con **út**, mỗi con trỏ tới **anh** ngay trước, rồi decode xong thì đảo list con) — miễn decode làm đúng phép ngược của encode.

In [21]:
def encode_rl(node):
    if node is None:
        return None
    b = TreeNode(node.val)
    prev = None
    for child in node.children or []:
        bc = encode_rl(child)
        if prev is None:
            b.right = bc                     # con đầu -> RIGHT
        else:
            prev.left = bc                   # em kế -> LEFT
        prev = bc
    return b


def decode_rl(b):
    if b is None:
        return None
    node = Node(b.val, [])
    cur = b.right
    while cur is not None:
        node.children.append(decode_rl(cur))
        cur = cur.left
    return node


def mirror(root):
    if root is None:
        return None
    m = TreeNode(root.val)
    m.left, m.right = mirror(root.right), mirror(root.left)
    return m


e1_rl = encode_rl(build_nary(ex1))
print("biến thể, Example 1:", tree_to_list(e1_rl))
print_tree(e1_rl)
print("là gương của cách chính?", same_tree(e1_rl, mirror(encode_lcrs(build_nary(ex1)))))
for vals in [ex1, ex2, ex3]:
    assert nary_to_list(decode_rl(encode_rl(build_nary(vals)))) == vals
print("round-trip biến thể: OK")

biến thể, Example 1: [1, None, 3, 2, 5, 4, None, 6]
1
├── L: ·
└── R: 3
    ├── L: 2
    │   ├── L: 4
    │   └── R: ·
    └── R: 5
        ├── L: 6
        └── R: ·
là gương của cách chính? True
round-trip biến thể: OK


## 10. Cách 4 — Bản BFS lặp bằng queue (không đệ quy)

### 10.1 Vì sao cần bản lặp?

Bản đệ quy ở mục 6 sâu tối đa `H` tầng. Đề cho `H <= 1000`, trong khi giới hạn đệ quy mặc định của Python cũng khoảng `1000` (và bộ chấm còn chiếm thêm vài tầng). Thực tế LeetCode thường nâng giới hạn này nên bản đệ quy vẫn qua, nhưng một bản **không đệ quy** thì yên tâm tuyệt đối, và là kỹ năng cần có khi phỏng vấn.

Ý tưởng: vẫn đúng **quy tắc LCRS**, chỉ đổi **thứ tự** dựng node — từ "đệ quy theo chiều sâu" sang "từng tầng một" bằng queue (BFS — Breadth-First Search, duyệt theo chiều rộng).

**Encode BFS:** queue chứa các **cặp** `(node N-ary, node nhị phân tương ứng)` đã tạo nhưng chưa nối con.

1. Tạo `B(root)`, đưa cặp `(root, B(root))` vào queue.
2. Lặp: `popleft` một cặp `(x, bx)`. Với các con `c1..ck` của `x`: tạo `B(ci)`, nối `bx.left = B(c1)` và `B(c_{i-1}).right = B(ci)` (y hệt cách chính), rồi đưa cặp `(ci, B(ci))` vào queue để tới lượt nó nối con.
3. Queue rỗng thì xong.

Vì mỗi cặp vào queue đúng một lần, mỗi node được tạo và nối đúng một lần. Kết quả phải **giống hệt** cây của bản đệ quy (chỉ khác thứ tự tạo node) — cell dưới kiểm chứng bằng `same_tree`.

In [22]:
def encode_bfs(root):
    if root is None:
        return None
    broot = TreeNode(root.val)
    queue = deque([(root, broot)])
    while queue:
        x, bx = queue.popleft()
        prev = None
        for child in x.children or []:
            bc = TreeNode(child.val)
            if prev is None:
                bx.left = bc                 # con đầu -> left
            else:
                prev.right = bc              # em kế -> right
            prev = bc
            queue.append((child, bc))        # tới lượt child nối con của nó
    return broot


for vals in [ex1, ex2, ex3, hard]:
    t = build_nary(vals)
    assert same_tree(encode_bfs(t), encode_lcrs(t))
print("encode_bfs cho cây nhị phân GIỐNG HỆT encode_lcrs trên 4 cây")
print(tree_to_list(encode_bfs(build_nary(ex1))))

encode_bfs cho cây nhị phân GIỐNG HỆT encode_lcrs trên 4 cây
[1, 3, None, 5, 2, None, 6, None, 4]


### 10.2 Trace encode BFS trên Example 1

| Lượt | `popleft` | Các con | Nối | Queue sau lượt |
|---:|---|---|---|---|
| 1 | `(1, B1)` | `[3, 2, 4]` | `B1.left=B3`, `B3.right=B2`, `B2.right=B4` | `[3, 2, 4]` |
| 2 | `(3, B3)` | `[5, 6]` | `B3.left=B5`, `B5.right=B6` | `[2, 4, 5, 6]` |
| 3 | `(2, B2)` | `[]` | — | `[4, 5, 6]` |
| 4 | `(4, B4)` | `[]` | — | `[5, 6]` |
| 5 | `(5, B5)` | `[]` | — | `[6]` |
| 6 | `(6, B6)` | `[]` | — | `[]` → xong |

So với trace đệ quy (mục 7.1): cùng 5 phép nối, chỉ khác thứ tự. Ví dụ `B3.right = B2` ở đây xảy ra **trước** `B3.left = B5`; ở bản đệ quy thì ngược lại. Kết quả cuối cùng như nhau vì mỗi phép nối chỉ phụ thuộc vào quan hệ cha-con, anh-em — không phụ thuộc thứ tự thực hiện.

In [23]:
def trace_encode_bfs(root):
    broot = TreeNode(root.val)
    queue = deque([(root, broot)])
    turn = 0
    while queue:
        x, bx = queue.popleft()
        turn += 1
        links, prev = [], None
        for child in x.children or []:
            bc = TreeNode(child.val)
            if prev is None:
                bx.left = bc
                links.append(f"B{x.val}.left=B{child.val}")
            else:
                prev.right = bc
                links.append(f"B{prev.val}.right=B{child.val}")
            prev = bc
            queue.append((child, bc))
        print(f"lượt {turn}: pop {x.val} | con {[c.val for c in x.children or []]} | "
              f"nối {', '.join(links) or '-'} | queue {[p[0].val for p in queue]}")
    return broot


print(tree_to_list(trace_encode_bfs(build_nary(ex1))))

lượt 1: pop 1 | con [3, 2, 4] | nối B1.left=B3, B3.right=B2, B2.right=B4 | queue [3, 2, 4]
lượt 2: pop 3 | con [5, 6] | nối B3.left=B5, B5.right=B6 | queue [2, 4, 5, 6]
lượt 3: pop 2 | con [] | nối - | queue [4, 5, 6]
lượt 4: pop 4 | con [] | nối - | queue [5, 6]
lượt 5: pop 5 | con [] | nối - | queue [6]
lượt 6: pop 6 | con [] | nối - | queue []
[1, 3, None, 5, 2, None, 6, None, 4]


### 10.3 Decode BFS bằng queue

Đối xứng với encode: queue chứa các cặp `(node nhị phân, node N-ary tương ứng)`.

1. Tạo `N(root)` từ `data`, đưa `(data, N(root))` vào queue.
2. Lặp: `popleft` một cặp `(b, x)`. Đi dọc linked list các con: `cur = b.left`; trong khi `cur` khác `None`: tạo `Node(cur.val, [])`, `append` vào `x.children`, đưa cặp `(cur, node mới)` vào queue, rồi `cur = cur.right`.
3. Queue rỗng thì xong.

Cả encode lẫn decode BFS đều **không có lời gọi đệ quy nào**, nên chạy được với cây sâu bao nhiêu cũng được (chỉ tốn bộ nhớ queue, tối đa cỡ một tầng `W`). Test trên đường thẳng sâu `50 000` tầng — gấp 50 lần mức đề cho.

In [24]:
def decode_bfs(data):
    if data is None:
        return None
    root = Node(data.val, [])
    queue = deque([(data, root)])
    while queue:
        b, x = queue.popleft()
        cur = b.left                         # đầu linked list các con
        while cur is not None:
            child = Node(cur.val, [])
            x.children.append(child)
            queue.append((cur, child))       # tới lượt child nhận con
            cur = cur.right                  # sang em kế
    return root


for vals in [ex1, ex2, ex3, hard]:
    assert nary_to_list(decode_bfs(encode_bfs(build_nary(vals)))) == vals
deep = make_path(50_000)
assert same_nary(decode_bfs(encode_bfs(deep)), deep)
wide = make_star(9999)
assert same_nary(decode_bfs(encode_bfs(wide)), wide)
print("BFS round-trip OK: 4 cây nhỏ + đường sâu 50 000 + sao 9999 con")
print("decode BFS của Example 2:", nary_to_list(decode_bfs(encode_bfs(build_nary(ex2)))))

BFS round-trip OK: 4 cây nhỏ + đường sâu 50 000 + sao 9999 con
decode BFS của Example 2: [1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14]


### 10.4 Bản nộp LeetCode của cách BFS

Trong notebook đặt tên `CodecBFS` để khỏi ghi đè `Codec` ở mục 8; khi nộp LeetCode thì đổi lại tên thành `Codec`.

In [25]:
class CodecBFS:                     # khi nộp LeetCode: đổi tên thành Codec
    # Encodes an n-ary tree to a binary tree.
    def encode(self, root: 'Optional[Node]') -> Optional[TreeNode]:
        if root is None:
            return None
        broot = TreeNode(root.val)
        queue = deque([(root, broot)])
        while queue:
            x, bx = queue.popleft()
            prev = None
            for child in x.children or []:
                bc = TreeNode(child.val)
                if prev is None:
                    bx.left = bc
                else:
                    prev.right = bc
                prev = bc
                queue.append((child, bc))
        return broot

    # Decodes your binary tree to an n-ary tree.
    def decode(self, data: Optional[TreeNode]) -> 'Optional[Node]':
        if data is None:
            return None
        root = Node(data.val, [])
        queue = deque([(data, root)])
        while queue:
            b, x = queue.popleft()
            cur = b.left
            while cur is not None:
                child = Node(cur.val, [])
                x.children.append(child)
                queue.append((cur, child))
                cur = cur.right
        return root


for vals in [ex1, ex2, ex3, [0], nary_to_list(make_star(9999)), nary_to_list(make_path(1000))]:
    assert run_leetcode(CodecBFS, vals) == vals
print("CodecBFS: PASS 3 ví dụ + 1 node + sao 9999 con + đường sâu 1000")

CodecBFS: PASS 3 ví dụ + 1 node + sao 9999 con + đường sâu 1000


## 11. Cây "rộng" và cây "sâu": đệ quy sai chỗ là vỡ

### 11.1 Chiều cao cây nhị phân sau encode

Trong LCRS, đi xuống một thế hệ = một cạnh `left`, đi sang em = một cạnh `right`. Một đường đi trong cây nhị phân có thể đi sang em rất nhiều lần, nên **chiều cao cây nhị phân có thể lớn hơn `H` rất nhiều**:

- Cây hình sao 9999 con: `H = 2`, nhưng cây nhị phân là `B(0).left = B(1)`, `B(1).right = B(2)`, ... — một chuỗi **cao 10 000**.
- Đường thẳng sâu `d`: `H = d`, cây nhị phân là chuỗi `left` cao `d`.

Vì thế, **hàm đệ quy theo kiểu cây nhị phân thông thường** (gọi đệ quy cho cả `left` lẫn `right`) sẽ sâu tới `chiều cao nhị phân`, có thể ~`10^4` với input hợp lệ → `RecursionError`. Cách viết ở mục 6 **đệ quy theo `left`, lặp theo `right`** nên chỉ sâu `H`.

### 11.2 Demo: bản encode/decode "đệ quy cả sang em"

Một cách viết rất tự nhiên — và sai — là `decode_naive(b)` trả về **cả nhóm anh em** bắt đầu từ `b`: `[node của b] + decode_naive(b.right)`. Nó đúng trên cây nhỏ, nhưng đệ quy sang em làm call stack sâu bằng **số anh em**. Cell dưới thử trên cây hình sao 9999 con (input hợp lệ theo đề).

In [26]:
def decode_naive_siblings(b):
    # trả về LIST các node N-ary của cả nhóm anh em bắt đầu từ b  (đệ quy cả sang em!)
    if b is None:
        return []
    node = Node(b.val, decode_naive_siblings(b.left))
    return [node] + decode_naive_siblings(b.right)


small = encode_lcrs(build_nary(ex1))
print("cây nhỏ, bản ngây thơ:", nary_to_list(decode_naive_siblings(small)[0]))

star = make_star(9999)
bstar = encode_lcrs(star)
print("sao 9999 con: H =", height_nary(star), "| chiều cao nhị phân =", height_bin(bstar))
try:
    decode_naive_siblings(bstar)
except RecursionError as e:
    print("decode_naive_siblings ->", type(e).__name__)
print("decode_lcrs (lặp theo em) ->", same_nary(decode_lcrs(bstar), star))

path = make_path(20_000)               # vượt mức đề (H <= 1000), chỉ để thấy giới hạn của bản đệ quy
try:
    encode_lcrs(path)
except RecursionError as e:
    print("đường sâu 20000: encode_lcrs ->", type(e).__name__, "| encode_bfs ->",
          same_nary(decode_bfs(encode_bfs(path)), path))

cây nhỏ, bản ngây thơ: [1, None, 3, 2, 4, None, 5, 6]
sao 9999 con: H = 2 | chiều cao nhị phân = 10000
decode_naive_siblings -> RecursionError
decode_lcrs (lặp theo em) -> True
đường sâu 20000: encode_lcrs -> RecursionError | encode_bfs -> True


## 12. Đối chiếu chéo mọi cách trên cây N-ary ngẫu nhiên có `seed`

Sinh cây ngẫu nhiên bằng `random_nary(n, max_val, rng)`: lần lượt tạo node thứ `1, 2, ..., n-1`, mỗi node chọn **ngẫu nhiên một cha** trong các node đã có và được thêm vào **cuối** list con của cha. Cách này sinh đủ loại hình dạng: cây sâu, cây rộng, node nhiều con, lá kẹp giữa. Giá trị lấy trong `[0, max_val]`, với `max_val` nhỏ để có **nhiều giá trị trùng** và nhiều số `0` (thứ hay làm lộ bug).

Với mỗi cây, kiểm tra:

1. Round-trip đúng với cả 4 cách: lười, LCRS đệ quy, biến thể, BFS (so bằng `same_nary` **và** bằng list level-order).
2. LCRS đệ quy và BFS cho cây nhị phân **giống hệt** nhau; biến thể là **gương** của LCRS.
3. Các tính chất ở mục 5.2: số node nhị phân của LCRS bằng `n`, root nhị phân không có con phải, cách lười dùng `2n` node.
4. `Codec` và `CodecBFS` qua `run_leetcode`.

`random.Random(seed)` tạo bộ sinh số ngẫu nhiên riêng với `seed` cố định — chạy lại bao nhiêu lần cũng ra đúng các cây đó, nên lỗi (nếu có) luôn tái hiện được.

In [27]:
def random_nary(n, max_val, rng):
    if n == 0:
        return None
    nodes = [Node(rng.randint(0, max_val), [])]
    for _ in range(1, n):
        child = Node(rng.randint(0, max_val), [])
        rng.choice(nodes).children.append(child)      # gắn vào cuối list con của một cha ngẫu nhiên
        nodes.append(child)
    return nodes[0]


rng = random.Random(431)
methods = {
    "lazy": (encode_lazy, decode_lazy),
    "lcrs": (encode_lcrs, decode_lcrs),
    "rl":   (encode_rl, decode_rl),
    "bfs":  (encode_bfs, decode_bfs),
}
trials, total_nodes = 0, 0
for _ in range(400):
    n = rng.randint(0, 40)
    t = random_nary(n, rng.choice([0, 3, 10, 10_000]), rng)
    vals = nary_to_list(t)
    for name, (enc, dec) in methods.items():
        back = dec(enc(t))
        assert same_nary(back, t) and nary_to_list(back) == vals, name
    b = encode_lcrs(t)
    assert same_tree(b, encode_bfs(t)) and same_tree(encode_rl(t), mirror(b))
    assert count_bin(b) == n and count_bin(encode_lazy(t)) == 2 * n
    assert b is None or b.right is None
    assert run_leetcode(Codec, vals) == vals and run_leetcode(CodecBFS, vals) == vals
    trials += 1
    total_nodes += n
print(f"{trials} cây ngẫu nhiên ({total_nodes} node tổng cộng): mọi cách đều khớp nhau")

big = random_nary(10_000, 10_000, random.Random(2024))
print("cây ngẫu nhiên 10^4 node: H =", height_nary(big),
      "| Codec:", same_nary(Codec().decode(Codec().encode(big)), big),
      "| CodecBFS:", same_nary(CodecBFS().decode(CodecBFS().encode(big)), big))

400 cây ngẫu nhiên (7852 node tổng cộng): mọi cách đều khớp nhau


cây ngẫu nhiên 10^4 node: H = 26 | Codec: True | CodecBFS: True


## 13. Vì sao cách chính đúng? Độ phức tạp và so sánh các cách

### 13.1 Bất biến (invariant) của LCRS

Gọi `B(x)` là node nhị phân của `x`. Encode bảo đảm bất biến sau cho **mọi** node `x` có các con `c1, c2, ..., ck`:

> `B(x).left = B(c1)` (hoặc `None` nếu `k = 0`), và `B(ci).right = B(c_{i+1})` với `i < k`, `B(ck).right = None`.
> Ngoài ra `B(root).right = None`.

Nói bằng lời: **từ `B(x)`, đi trái một bước rồi đi phải `i - 1` bước sẽ tới đúng con thứ `i` của `x`, và đi phải thêm nữa thì gặp `None` đúng lúc hết con.**

Bất biến này đúng vì mỗi node N-ary `c` có **đúng một** "vị trí" trong cây: hoặc là con đầu của cha (→ là `left` của `B(cha)`), hoặc là em của một anh duy nhất (→ là `right` của `B(anh)`). Không có node nào được gán vào hai ô, không ô nào nhận hai node.

### 13.2 Decode là phép ngược của encode (quy nạp theo chiều cao)

- Cây rỗng: `encode(None) = None`, `decode(None) = None`. Đúng.
- Giả sử decode đúng với mọi cây cao `< h`. Xét `x` cao `h`, có con `c1..ck` (mỗi con cao `< h`). `decode(B(x))` tạo `Node(x.val)`, rồi theo bất biến, con trỏ `cur` lần lượt bằng `B(c1), B(c2), ..., B(ck)` rồi `None` — **đúng `k` lần, đúng thứ tự**. Mỗi lần gọi `decode(B(ci))`, theo giả thiết quy nạp, cho lại đúng cây con `ci` (để ý: `decode(B(ci))` chỉ đọc `B(ci).left` và phần bên dưới, **không** đọc `B(ci).right` — phần anh em là việc của cha). Vậy `decode(B(x))` cho đúng cây `x`. ∎

Thêm nữa, mọi cây nhị phân có root không có con phải đều là encode của **đúng một** cây N-ary (đọc ngược bất biến). Đây là một **song ánh** (bijection) nổi tiếng: cây có thứ tự `n` node ↔ cây nhị phân `n` node có root không có con phải. Hệ quả thú vị: số cây có thứ tự `n` node bằng số cây nhị phân `n - 1` node, tức số **Catalan** `C(n-1)` — cell dưới kiểm chứng bằng cách vét cạn với `n` nhỏ.

### 13.3 Hệ quả: đọc cấu trúc N-ary ngay trên cây nhị phân

- **Độ sâu N-ary** của một node = số cạnh `left` trên đường từ root nhị phân tới nó (cạnh `right` không đổi thế hệ).
- Cell dưới kiểm chứng điều này trên cây ngẫu nhiên, cùng với phép đếm Catalan.

In [28]:
from math import comb


def all_shapes(n):
    # mọi cây có thứ tự (chỉ xét hình dạng) với đúng n node, liệt kê dạng tuple lồng nhau
    if n == 1:
        return [()]
    out = []
    def forests(m):           # mọi "rừng" (list cây con có thứ tự) với tổng m node
        if m == 0:
            return [()]
        res = []
        for first in range(1, m + 1):
            for t in all_shapes(first):
                for rest in forests(m - first):
                    res.append((t,) + rest)
        return res
    return forests(n - 1)


def to_node(shape):
    return Node(0, [to_node(s) for s in shape])


for n in range(1, 8):
    shapes = all_shapes(n)
    encodings = {tuple(tree_to_list(encode_lcrs(to_node(s)))) for s in shapes}
    catalan = comb(2 * (n - 1), n - 1) // n
    assert len(encodings) == len(shapes) == catalan          # encode không làm 2 cây trùng nhau
    print(f"n={n}: {len(shapes):>3} cây có thứ tự -> {len(encodings):>3} cây nhị phân khác nhau (Catalan C({n-1}) = {catalan})")


def check_depth_equals_left_edges(root):
    # độ sâu N-ary (tính bằng BFS trên N-ary) phải bằng số cạnh left trên cây nhị phân
    depth_nary = {}
    queue = deque([(root, 0)])
    while queue:
        x, d = queue.popleft()
        depth_nary[id(x)] = d
        queue.extend((c, d + 1) for c in x.children)
    nodes_in_order = []                                   # pre-order N-ary = pre-order nhị phân (mục 15)
    stack = [root]
    while stack:
        x = stack.pop()
        nodes_in_order.append(x)
        stack.extend(reversed(x.children))
    lefts, stack = [], [(encode_lcrs(root), 0)]
    while stack:
        b, k = stack.pop()
        lefts.append(k)
        if b.right: stack.append((b.right, k))            # sang em: không đổi thế hệ
        if b.left:  stack.append((b.left, k + 1))         # xuống con: +1 thế hệ
    return [depth_nary[id(x)] for x in nodes_in_order] == lefts


rng = random.Random(7)
print("độ sâu N-ary = số cạnh left:", all(check_depth_equals_left_edges(random_nary(rng.randint(1, 60), 9, rng)) for _ in range(200)))

n=1:   1 cây có thứ tự ->   1 cây nhị phân khác nhau (Catalan C(0) = 1)
n=2:   1 cây có thứ tự ->   1 cây nhị phân khác nhau (Catalan C(1) = 1)
n=3:   2 cây có thứ tự ->   2 cây nhị phân khác nhau (Catalan C(2) = 2)
n=4:   5 cây có thứ tự ->   5 cây nhị phân khác nhau (Catalan C(3) = 5)
n=5:  14 cây có thứ tự ->  14 cây nhị phân khác nhau (Catalan C(4) = 14)
n=6:  42 cây có thứ tự ->  42 cây nhị phân khác nhau (Catalan C(5) = 42)
n=7: 132 cây có thứ tự -> 132 cây nhị phân khác nhau (Catalan C(6) = 132)
độ sâu N-ary = số cạnh left: True


### 13.4 Độ phức tạp

Mỗi cách đi qua mỗi node đúng một lần (mỗi node được tạo một lần, mỗi tham chiếu được gán một lần), nên **thời gian đều là `O(n)`** cho cả encode lẫn decode. Khác biệt nằm ở bộ nhớ phụ và chất lượng cây nhị phân:

| Cách | Số node nhị phân | Chiều cao nhị phân | Bộ nhớ phụ (ngoài kết quả) | Đúng tinh thần đề? | Ghi chú |
|---|---|---|---|---|---|
| 1. Lười (dãy số → chuỗi `right`) | `2n` | `2n` | `O(n)` mảng token + `O(H)` đệ quy | Không | thực chất là bài 428 đội lốt `TreeNode` |
| 2. LCRS đệ quy (cách chính) | `n` | tới `n` | `O(H)` call stack, `H <= 1000` | Có | ngắn, đẹp, dễ chứng minh |
| 3. Biến thể (con đầu → `right`) | `n` | tới `n` | `O(H)` | Có | gương của cách 2 |
| 4. LCRS bằng BFS | `n` | tới `n` | `O(W)` queue (`W` = tầng rộng nhất, tới `n`) | Có | không đệ quy, an toàn với mọi độ sâu |

Ghi chú về Big-O:

- `O(H)` của cách 2 là **worst case theo chiều cao**: cây cân bằng nhiều nhánh thì `H` rất nhỏ; đường thẳng thì `H = n`, nhưng đề chặn `H <= 1000`.
- `O(W)` của BFS: cây hình sao có `W = n - 1`, nên worst case vẫn là `O(n)`. Không có cách nào "tốt hơn hẳn" về bộ nhớ trong mọi trường hợp: đệ quy tốn theo chiều sâu, BFS tốn theo chiều rộng.
- Kết quả trả về (cây nhị phân `n` node, hoặc cây N-ary `n` node) luôn tốn `O(n)` — đó là output, không tính vào bộ nhớ phụ.

## 14. Lỗi thường gặp

### Lỗi 1: Lưu trạng thái trong `self` (vi phạm stateless)

```python
class Codec:
    def encode(self, root):
        self.saved = root          # SAI: giấu cây trong object
        return TreeNode(0)
    def decode(self, data):
        return self.saved
```

Chạy với cùng một object thì "qua", nhưng đề cấm rõ ràng, và một object decode khác (hay một máy khác) sẽ không có `saved`. Cây nhị phân phải **tự chứa đủ thông tin**. `run_leetcode` bắt lỗi này vì decode bằng object mới (demo ở cell dưới).

### Lỗi 2: Đặt con thứ 1 vào `left`, con thứ 2 vào `right`, bỏ qua phần còn lại

Đó là phản xạ "cây nhị phân = 2 con". Mất dữ liệu ngay khi một node có 3 con (Example 1). Và ngay cả với node 2 con, decode không phân biệt được "`right` là con thứ 2" với... chẳng có gì để phân biệt. Phải đổi **ý nghĩa** của ô `right` thành "em kế".

### Lỗi 3: Nối em vào `right` của **cha** thay vì của **anh**

```python
prev = b                       # SAI: prev khởi đầu là chính b
for child in node.children:
    bc = enc(child)
    prev.right = bc            # con đầu bị gắn vào b.right (vị trí của "em của b")!
    prev = bc
```

Con đầu phải vào `b.left`; chỉ các con **sau** mới vào `right` của **anh ngay trước**. Bug này làm root nhị phân có con phải và **không có** con trái, nên decode đúng chuẩn chỉ đọc ra được mỗi root `[1]` — toàn bộ con cháu biến mất (xem cell dưới).

### Lỗi 4: `Node(val)` rồi `.children.append(...)`

`Node(5)` để `children = None`, nên `node.children.append(x)` ném `AttributeError: 'NoneType' object has no attribute 'append'`. Luôn tạo `Node(val, [])`. Ngược lại, ở **encode**, input của LeetCode có thể có `children = None` — viết `for c in node.children or []`, đừng viết `for c in node.children`.

### Lỗi 5: Đệ quy cả sang em (`right`)

Đúng trên cây nhỏ nhưng `RecursionError` với cây hình sao 9999 con — input hợp lệ (mục 11.2). Quy tắc: **đệ quy theo `left` (xuống thế hệ), lặp theo `right` (sang anh em)**.

### Lỗi 6: Quên `cur = cur.right` trong vòng `while` của decode

Vòng `while cur is not None` không bao giờ dừng (lặp vô hạn, cứ decode mãi con đầu). Mỗi vòng `while` đi dọc linked list phải có đúng một dòng "tiến con trỏ".

### Lỗi 7: Dùng kiểm tra "truthy" với giá trị

`if node.val:` hay `if values[i]:` coi `0` như "không có" — trong khi `0 <= val` là hợp lệ. Luôn so `is None` / `is not None`. Tương tự, đừng dùng `val` để nhận diện node (giá trị có thể trùng); dùng chính object hoặc `id(...)`.

### Lỗi 8: Đổi thứ tự con

Ví dụ duyệt `reversed(node.children)` khi encode nhưng decode lại `append` theo thứ tự xuôi, hay dùng `set` để chứa con. Cây `[1,null,3,2,4]` thành `[1,null,4,2,3]` — sai, vì thứ tự con là một phần của cấu trúc.

In [29]:
# Lỗi 1: codec có trạng thái
class CodecStateful:
    def encode(self, root):
        self.saved = root
        return TreeNode(0)

    def decode(self, data):
        return self.saved


c = CodecStateful()
print("Lỗi 1 - cùng object  :", nary_to_list(c.decode(c.encode(build_nary(ex1)))))   # trông như đúng
try:
    run_leetcode(CodecStateful, ex1)
except AttributeError as e:
    print("Lỗi 1 - object mới   : AttributeError ->", e)


# Lỗi 2: con 1 -> left, con 2 -> right
def encode_two_slots(node):
    if node is None:
        return None
    b = TreeNode(node.val)
    kids = node.children or []
    if len(kids) > 0: b.left = encode_two_slots(kids[0])
    if len(kids) > 1: b.right = encode_two_slots(kids[1])     # kids[2:] bị bỏ rơi
    return b

print("Lỗi 2 - số node còn lại:", count_bin(encode_two_slots(build_nary(ex1))), "/", count_nary(build_nary(ex1)))


# Lỗi 3: nối con đầu vào right của cha
def encode_bug_parent_right(node):
    if node is None:
        return None
    b = TreeNode(node.val)
    prev = b
    for child in node.children or []:
        bc = encode_bug_parent_right(child)
        prev.right = bc
        prev = bc
    return b

bad = encode_bug_parent_right(build_nary(ex1))
print("Lỗi 3 - cây nhị phân:", tree_to_list(bad), "-> decode ra:", nary_to_list(decode_lcrs(bad)))


# Lỗi 4: Node(val) rồi append
try:
    leaf = Node(5)
    leaf.children.append(Node(6))
except AttributeError as e:
    print("Lỗi 4 -", type(e).__name__, "->", e)
print("Lỗi 4 - encode node có children=None vẫn chạy nhờ `or []`:", tree_to_list(encode_lcrs(Node(5))))

Lỗi 1 - cùng object  : [1, None, 3, 2, 4, None, 5, 6]
Lỗi 1 - object mới   : AttributeError -> 'CodecStateful' object has no attribute 'saved'
Lỗi 2 - số node còn lại: 5 / 6
Lỗi 3 - cây nhị phân: [1, None, 3, None, 2, None, 4] -> decode ra: [1]
Lỗi 4 - AttributeError -> 'NoneType' object has no attribute 'append'
Lỗi 4 - encode node có children=None vẫn chạy nhờ `or []`: [5]


## 15. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu nên nghĩ tới left-child right-sibling

- Cần biểu diễn cây **nhiều con** bằng cấu trúc chỉ có **2 con trỏ** (cây nhị phân, hay node có `next` + `child` như LeetCode 430 Flatten a Multilevel Doubly Linked List).
- Cần lưu **rừng** (nhiều cây) thành một cây: nối các root bằng `right` (bài tập 6).
- Thấy "danh sách các con" mà muốn xử lý bằng linked list: con đầu là `head`, anh em nối bằng `next`.

### Sự thật bất ngờ về thứ tự duyệt

Với cây nhị phân LCRS `B` của cây N-ary `T`:

- **pre-order(T) = pre-order(B)**: pre-order N-ary ghi node rồi lần lượt các cây con; pre-order nhị phân ghi node, rồi `left` (= cả họ hàng con cháu), rồi `right` (= các em và con cháu của chúng) — cùng một thứ tự.
- **post-order(T) = in-order(B)**: in-order nhị phân = `left` (toàn bộ con cháu) → node → `right` (các em), đúng là "con cháu trước, mình sau, rồi tới em" của post-order N-ary.

Cell dưới kiểm chứng trên cây ngẫu nhiên. Hệ quả: các bài duyệt N-ary (589, 590) có thể giải bằng code duyệt cây nhị phân quen thuộc sau khi encode.

### Khung tư duy nhớ nhanh

```text
left  = "xuống": con đầu      (đệ quy / đưa vào queue)
right = "ngang": em kế         (vòng lặp dọc linked list)
```

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 428. Serialize and Deserialize N-ary Tree (Hard, Premium) | cùng cây N-ary, nhưng encode thành **chuỗi** — chính là bước 1 của cách lười (bài tập 5) |
| 297. Serialize and Deserialize Binary Tree (Hard) | serialize cây nhị phân; kết hợp với 431 → serialize được cây N-ary |
| 589 / 590. N-ary Tree Preorder / Postorder Traversal | trùng pre-order / in-order của cây LCRS |
| 429. N-ary Tree Level Order Traversal | BFS bằng queue như mục 10 |
| 559. Maximum Depth of N-ary Tree | `height_nary`; trên cây LCRS = số cạnh `left` lớn nhất + 1 (bài tập 4) |
| 430. Flatten a Multilevel Doubly Linked List | cấu trúc `child` + `next` chính là LCRS |
| 1490. Clone N-ary Tree | đệ quy / BFS tạo bản sao node từng node như encode |

In [30]:
def preorder_bin(root):
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node.val)
        if node.right: stack.append(node.right)
        if node.left: stack.append(node.left)
    return out


def inorder_bin(root):
    out, stack, cur = [], [], root
    while stack or cur:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        out.append(cur.val)
        cur = cur.right
    return out


t = build_nary(ex1)
print("pre-order N-ary :", preorder_nary(t), "| pre-order nhị phân:", preorder_bin(encode_lcrs(t)))
print("post-order N-ary:", postorder_nary(t), "| in-order nhị phân :", inorder_bin(encode_lcrs(t)))
rng = random.Random(589)
for _ in range(300):
    t = random_nary(rng.randint(0, 50), 20, rng)
    b = encode_lcrs(t)
    assert preorder_nary(t) == preorder_bin(b) and postorder_nary(t) == inorder_bin(b)
print("300 cây ngẫu nhiên: pre=pre, post=in -> đúng")

pre-order N-ary : [1, 3, 5, 6, 2, 4] | pre-order nhị phân: [1, 3, 5, 6, 2, 4]
post-order N-ary: [5, 6, 3, 2, 4, 1] | in-order nhị phân : [5, 6, 3, 2, 4, 1]
300 cây ngẫu nhiên: pre=pre, post=in -> đúng


## 16. Bài tập vận dụng

Làm theo thứ tự. Gợi ý ở mục 17, đáp án ở mục 18 (cuối notebook). Các cell TODO dưới đây chỉ **định nghĩa** hàm; test được để sẵn dưới dạng comment — làm xong thì bỏ comment để tự kiểm tra.

### Bài 1 — Đoán output (không chạy code)

Cây N-ary `[1,null,2,3,null,4]`. Hãy vẽ cây, rồi đoán:

1. `tree_to_list(encode_lcrs(build_nary([1,None,2,3,None,4])))` in ra gì?
2. Cây nhị phân đó cao bao nhiêu? Có mấy node?
3. Với **biến thể** (cách 3), list level-order của cây nhị phân là gì?

### Bài 2 — Tính tay decode

Cho cây nhị phân (đã được encode theo LCRS) `[1,2,null,3,4,null,5,null,6]`. Hãy vẽ cây nhị phân, rồi **decode bằng tay**: viết cây N-ary dạng list level-order của LeetCode. Gợi ý: đi trái là xuống con cả, đi phải là sang em.

### Bài 3 — Điền TODO: lời giải chính

Hoàn thành class dưới đây (đúng cách 2) mà không nhìn lại mục 8.

In [31]:
class CodecExercise:
    def encode(self, root):
        # TODO 1: nếu root là None -> trả None
        # TODO 2: hàm lồng enc(node): tạo TreeNode(node.val), prev = None
        # TODO 3: duyệt node.children or []: bc = enc(child);
        #         con đầu -> b.left = bc, con sau -> prev.right = bc; cập nhật prev
        pass

    def decode(self, data):
        # TODO 4: nếu data là None -> trả None
        # TODO 5: hàm lồng dec(b): node = Node(b.val, []); cur = b.left
        # TODO 6: while cur: append dec(cur) vào node.children; cur = cur.right
        pass


# Bỏ comment để tự kiểm tra:
# for vals in [ex1, ex2, ex3]:
#     assert run_leetcode(CodecExercise, vals) == vals

### Bài 4 — Tự viết: đọc cấu trúc N-ary **ngay trên cây nhị phân** (không decode)

Cho `b` là node nhị phân trong một cây LCRS. Viết:

1. `children_vals(b)`: list giá trị các con (theo nghĩa N-ary) của node mà `b` biểu diễn, đúng thứ tự.
2. `nary_height_from_binary(root)`: chiều cao của cây N-ary (LeetCode 559), tính trực tiếp trên cây nhị phân `root`, **không** decode và **không** đệ quy (dùng stack). Nhớ: cạnh `left` +1 thế hệ, cạnh `right` thì không.

In [32]:
def children_vals(b):
    # TODO: bắt đầu từ b.left, đi dọc right, gom val
    raise NotImplementedError


def nary_height_from_binary(root):
    # TODO: stack các cặp (node, độ sâu N-ary); left -> độ sâu + 1, right -> giữ nguyên
    raise NotImplementedError


# Bỏ comment để tự kiểm tra:
# b = encode_lcrs(build_nary(ex2))
# assert children_vals(b) == [2, 3, 4, 5]
# assert children_vals(b.left.right) == [6, 7]        # node 3
# assert nary_height_from_binary(b) == 5
# assert nary_height_from_binary(None) == 0
# assert nary_height_from_binary(encode_lcrs(make_star(9999))) == 2

### Bài 5 — Biến thể: LeetCode 428. Serialize and Deserialize N-ary Tree

Lần này encode cây N-ary thành **chuỗi**. Viết `serialize_nary(root) -> str` và `deserialize_nary(data) -> Node`. Gợi ý định dạng: pre-order, mỗi node ghi `val` và `số con`, cách nhau bằng dấu phẩy (Example 1 → `"1,3,3,2,5,0,6,0,2,0,4,0"`); cây rỗng → `""`. Cẩn thận: `"".split(",")` cho `['']`, không phải `[]`.

In [33]:
def serialize_nary(root):
    # TODO: pre-order, ghi str(val) và str(số con)
    raise NotImplementedError


def deserialize_nary(data):
    # TODO: xử lý chuỗi rỗng; dùng iter để đọc (val, k) rồi dựng k con bằng đệ quy
    raise NotImplementedError


# Bỏ comment để tự kiểm tra:
# assert serialize_nary(build_nary(ex1)) == "1,3,3,2,5,0,6,0,2,0,4,0"
# for vals in [ex1, ex2, ex3]:
#     assert nary_to_list(deserialize_nary(serialize_nary(build_nary(vals)))) == vals

### Bài 6 — Mini challenge: encode cả một **rừng** (forest)

Một **rừng** là một list các cây N-ary (có thứ tự). Viết `encode_forest(roots) -> TreeNode` biến cả rừng thành **một** cây nhị phân, và `decode_forest(b) -> list[Node]` làm ngược lại. Yêu cầu: dùng ý tưởng LCRS, **không đệ quy** (dùng queue), rừng rỗng ↔ `None`.

Câu hỏi suy nghĩ thêm: sau khi giải xong, hãy giải thích vì sao mọi cây nhị phân (kể cả root **có** con phải) đều là encode của đúng một rừng.

In [34]:
def encode_forest(roots):
    # TODO: coi các root là "anh em" của nhau: root đầu là gốc nhị phân, root sau nối right
    #       mỗi cặp (node N-ary, node nhị phân) đưa vào queue để nối con như encode_bfs
    raise NotImplementedError


def decode_forest(b):
    # TODO: các root của rừng = b, b.right, b.right.right, ...; con thì decode như decode_bfs
    raise NotImplementedError


# Bỏ comment để tự kiểm tra:
# forest = [build_nary(ex1), Node(0, []), build_nary(ex2)]
# back = decode_forest(encode_forest(forest))
# assert len(back) == 3 and all(same_nary(x, y) for x, y in zip(back, forest))
# assert encode_forest([]) is None and decode_forest(None) == []

## 17. Gợi ý theo mức

Chỉ mở mức tiếp theo khi mức trước chưa đủ.

| Bài | Mức 1 — hướng tư duy | Mức 2 — cấu trúc / cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Root `1` có con `2, 3`; `2` có con `4`. | `B1.left = B2`, `B2.right = B3`, `B2.left = B4`. | Level-order nhị phân: tầng 0 `1`; tầng 1 `2, null`; tầng 2 là hai ô của `B2`. Biến thể = gương. |
| 2 | Vẽ cây nhị phân trước: `B2` có trái `B3`, phải `B4`. | Con của `x` = `B(x).left`, rồi cứ đi `right`. | Con của 1: `2, 4, 6`; con của 2: `3, 5`. |
| 3 | Đúng khung mục 8. | `prev is None` ↔ đang xử lý con đầu. | `node.children.append(dec(cur)); cur = cur.right`. |
| 4 | Con của `x` = linked list từ `b.left` nối bằng `right`. | Stack `[(root, 1)]`; mỗi lần pop cập nhật `best`. | `if b.left: push (b.left, d + 1)`; `if b.right: push (b.right, d)`. |
| 5 | Biết số con thì biết khi nào nhóm con kết thúc. | `",".join(tokens)`; `stream = iter(map(int, data.split(",")))`. | `val, k = next(s), next(s); return Node(val, [build() for _ in range(k)])`. |
| 6 | Các root trong rừng giống một nhóm anh em không có cha. | Dùng một node N-ary "cha giả" chứa cả rừng làm con, encode, rồi lấy `.left`. Hoặc tự nối các root bằng `right`. | Decode: `cur = b; while cur: tạo root, queue.append((cur, root)); cur = cur.right`, rồi BFS như `decode_bfs`. |

## 18. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
N-ary                nhị phân (LCRS)          biến thể (gương)
    1                     1                         1
   / \                   /                           \
  2   3                 2                             2
  |                    / \                           / \
  4                   4   3                         3   4
```

1. `[1, 2, None, 4, 3]`: tầng 0 là `1`; tầng 1 là hai ô của `B1`: `2, null`; tầng 2 là hai ô của `B2`: `4` (con cả của 2) và `3` (em của 2).
2. Cao **3**, có **4** node (đúng bằng `n`).
3. Biến thể là gương: `[1, None, 2, 3, 4]` (tầng 1: `null, 2`; tầng 2: `B2.left = 3` là em, `B2.right = 4` là con cả).

### Đáp án Bài 2

Cây nhị phân `[1,2,null,3,4,null,5,null,6]`:

```text
nhị phân                     N-ary
      1                          1
     /                        /  |  \
    2                        2   4   6
   / \                      / \
  3   4                    3   5
   \   \
    5   6
```

- Con của 1: bắt đầu `B1.left = B2`, đi phải `B4`, đi phải `B6`, rồi `None` → `[2, 4, 6]`.
- Con của 2: `B2.left = B3`, đi phải `B5` → `[3, 5]`. Các node `3, 4, 5, 6` có `left = None` nên là lá.
- Kết quả: `[1,null,2,4,6,null,3,5]`.

### Đáp án câu hỏi suy nghĩ của Bài 6

Trong một cây nhị phân bất kỳ, đi từ root dọc theo `right` ta được một dãy node — đó là các **root của rừng**. Với mỗi node, `left` là con cả của nó, và từ con cả đi dọc `right` là các con còn lại. Cách đọc này luôn xác định được và cho **đúng một** rừng; ngược lại encode rừng đó cho lại đúng cây nhị phân. Vậy *rừng có thứ tự `n` node ↔ cây nhị phân `n` node* là một song ánh. Cây N-ary trong bài 431 chỉ là trường hợp rừng có **một** cây — nên root nhị phân không có con phải.

Cell kế tiếp kiểm chứng Bài 1 và 2; các cell sau là đáp án code của Bài 3–6 kèm assert (Bài 6 có thêm phần kiểm chứng song ánh rừng ↔ cây nhị phân).

In [35]:
# Kiểm chứng Bài 1
t = build_nary([1, None, 2, 3, None, 4])
b = encode_lcrs(t)
print("Bài 1:", tree_to_list(b), "| cao", height_bin(b), "| số node", count_bin(b), "| biến thể:", tree_to_list(encode_rl(t)))
assert tree_to_list(b) == [1, 2, None, 4, 3] and tree_to_list(encode_rl(t)) == [1, None, 2, 3, 4]

# Kiểm chứng Bài 2
b2 = build_tree([1, 2, None, 3, 4, None, 5, None, 6])
ans2 = decode_lcrs(b2)
print("Bài 2:", nary_to_list(ans2))
print_nary(ans2)
assert nary_to_list(ans2) == [1, None, 2, 4, 6, None, 3, 5]

Bài 1: [1, 2, None, 4, 3] | cao 3 | số node 4 | biến thể: [1, None, 2, 3, 4]
Bài 2: [1, None, 2, 4, 6, None, 3, 5]
1
├── 2
│   ├── 3
│   └── 5
├── 4
└── 6


In [36]:
# ---------- Bài 3
class AnswerCodecExercise:
    def encode(self, root):
        if root is None:                                  # TODO 1
            return None

        def enc(node):                                    # TODO 2
            b = TreeNode(node.val)
            prev = None
            for child in node.children or []:             # TODO 3
                bc = enc(child)
                if prev is None:
                    b.left = bc
                else:
                    prev.right = bc
                prev = bc
            return b

        return enc(root)

    def decode(self, data):
        if data is None:                                  # TODO 4
            return None

        def dec(b):                                       # TODO 5
            node = Node(b.val, [])
            cur = b.left
            while cur is not None:                        # TODO 6
                node.children.append(dec(cur))
                cur = cur.right
            return node

        return dec(data)


for vals in [ex1, ex2, ex3]:
    assert run_leetcode(AnswerCodecExercise, vals) == vals
print("Bài 3: PASS")

Bài 3: PASS


In [37]:
# ---------- Bài 4
def answer_children_vals(b):
    out, cur = [], b.left
    while cur is not None:
        out.append(cur.val)
        cur = cur.right
    return out


def answer_nary_height_from_binary(root):
    best, stack = 0, [(root, 1)] if root else []
    while stack:
        b, d = stack.pop()
        best = max(best, d)
        if b.left:
            stack.append((b.left, d + 1))        # xuống con cả: +1 thế hệ
        if b.right:
            stack.append((b.right, d))           # sang em: cùng thế hệ
    return best


b = encode_lcrs(build_nary(ex2))
assert answer_children_vals(b) == [2, 3, 4, 5]
assert answer_children_vals(b.left.right) == [6, 7]
assert answer_nary_height_from_binary(b) == 5
assert answer_nary_height_from_binary(None) == 0
assert answer_nary_height_from_binary(encode_lcrs(make_star(9999))) == 2
rng = random.Random(559)
for _ in range(200):
    t = random_nary(rng.randint(0, 50), 9, rng)
    assert answer_nary_height_from_binary(encode_lcrs(t)) == height_nary(t)
print("Bài 4: PASS (con của root Example 2 =", answer_children_vals(b), ", chiều cao = 5)")

Bài 4: PASS (con của root Example 2 = [2, 3, 4, 5] , chiều cao = 5)


In [38]:
# ---------- Bài 5 (LeetCode 428)
def answer_serialize_nary(root):
    tokens = []

    def go(node):
        kids = node.children or []
        tokens.append(str(node.val))
        tokens.append(str(len(kids)))
        for c in kids:
            go(c)

    if root is not None:
        go(root)
    return ",".join(tokens)


def answer_deserialize_nary(data):
    if data == "":                                  # tránh bẫy "".split(",") == ['']
        return None
    stream = iter(map(int, data.split(",")))

    def build():
        val, k = next(stream), next(stream)
        return Node(val, [build() for _ in range(k)])

    return build()


assert answer_serialize_nary(build_nary(ex1)) == "1,3,3,2,5,0,6,0,2,0,4,0"
for vals in [ex1, ex2, ex3]:
    assert nary_to_list(answer_deserialize_nary(answer_serialize_nary(build_nary(vals)))) == vals
rng = random.Random(428)
for _ in range(200):
    t = random_nary(rng.randint(0, 40), 10_000, rng)
    assert same_nary(answer_deserialize_nary(answer_serialize_nary(t)), t)
print("Bài 5: PASS ->", answer_serialize_nary(build_nary(ex1)))

Bài 5: PASS -> 1,3,3,2,5,0,6,0,2,0,4,0


In [39]:
# ---------- Bài 6 (rừng <-> cây nhị phân)
def answer_encode_forest(roots):
    head = prev = None
    queue = deque()
    for r in roots:                                  # các root nối với nhau bằng right
        br = TreeNode(r.val)
        if prev is None:
            head = br
        else:
            prev.right = br
        prev = br
        queue.append((r, br))
    while queue:                                     # phần còn lại y hệt encode_bfs
        x, bx = queue.popleft()
        p = None
        for child in x.children or []:
            bc = TreeNode(child.val)
            if p is None:
                bx.left = bc
            else:
                p.right = bc
            p = bc
            queue.append((child, bc))
    return head


def answer_decode_forest(b):
    roots, queue, cur = [], deque(), b
    while cur is not None:                           # các root = b, b.right, b.right.right...
        r = Node(cur.val, [])
        roots.append(r)
        queue.append((cur, r))
        cur = cur.right
    while queue:                                     # phần còn lại y hệt decode_bfs
        bx, x = queue.popleft()
        c = bx.left
        while c is not None:
            child = Node(c.val, [])
            x.children.append(child)
            queue.append((c, child))
            c = c.right
    return roots


forest = [build_nary(ex1), Node(0, []), build_nary(ex2)]
back = answer_decode_forest(answer_encode_forest(forest))
assert len(back) == 3 and all(same_nary(x, y) for x, y in zip(back, forest))
assert answer_encode_forest([]) is None and answer_decode_forest(None) == []
rng = random.Random(1)
for _ in range(200):
    f = [random_nary(rng.randint(1, 15), 9, rng) for _ in range(rng.randint(0, 5))]
    g = answer_decode_forest(answer_encode_forest(f))
    assert len(g) == len(f) and all(same_nary(x, y) for x, y in zip(g, f))
# Mọi cây nhị phân là encode của đúng một rừng: decode rồi encode lại phải ra đúng cây nhị phân đó
for vals in [[1, 2, 3, None, None, 4, 5], [1, None, 2, None, 3], [7]]:
    bt = build_tree(vals)
    assert same_tree(answer_encode_forest(answer_decode_forest(bt)), bt)
print("Bài 6: PASS; cây nhị phân [1,2,3,null,null,4,5] là rừng có",
      len(answer_decode_forest(build_tree([1, 2, 3, None, None, 4, 5]))), "cây")
print("Đáp án tham khảo chạy PASS toàn bộ test!")

Bài 6: PASS; cây nhị phân [1,2,3,null,null,4,5] là rừng có 3 cây
Đáp án tham khảo chạy PASS toàn bộ test!


## 19. Template cuối cùng nên nhớ

**Left-child right-sibling (cách chính)** — `O(n)` time, `O(H)` bộ nhớ phụ:

```python
class Codec:
    def encode(self, root: 'Optional[Node]') -> Optional[TreeNode]:
        def enc(node):
            b = TreeNode(node.val)
            prev = None
            for child in node.children or []:   # anh em: VÒNG LẶP
                bc = enc(child)                 # xuống con: ĐỆ QUY
                if prev is None:
                    b.left = bc                 # con đầu -> left
                else:
                    prev.right = bc             # em kế  -> right của anh trước
                prev = bc
            return b
        return enc(root) if root is not None else None

    def decode(self, data: Optional[TreeNode]) -> 'Optional[Node]':
        def dec(b):
            node = Node(b.val, [])              # list rỗng THẬT
            cur = b.left
            while cur is not None:
                node.children.append(dec(cur))
                cur = cur.right                 # đừng quên tiến con trỏ
            return node
        return dec(data) if data is not None else None
```

**Câu thần chú**: *"Trái là xuống con cả, phải là sang em kế. Đệ quy theo trái, lặp theo phải."*

**Khi sợ đệ quy sâu**: BFS với queue chứa cặp `(node nguồn, node đích)` — cùng quy tắc nối, không đệ quy.

## 20. Checklist tự kiểm tra trước khi submit

- `encode` trả về **`TreeNode`** (hoặc `None`), không phải chuỗi hay `Node`?
- Cây rỗng: `encode(None) is None` và `decode(None) is None`?
- **Không** lưu gì trong `self` / biến global giữa `encode` và `decode` (stateless)?
- Con **đầu** vào `left` của **cha**; các con **sau** vào `right` của **anh ngay trước** (không phải của cha)?
- Encode chịu được `children = None` (`node.children or []`)? Decode tạo `Node(val, [])` với list thật?
- Vòng `while` của decode có `cur = cur.right`?
- Anh em xử lý bằng **vòng lặp**, chỉ đệ quy xuống con (cây hình sao 9999 con không làm tràn stack)? Hoặc dùng bản BFS?
- Giữ đúng **thứ tự** các con; không dùng kiểm tra "truthy" làm mất giá trị `0`?
- Đã test round-trip trên 3 ví dụ của đề + 1 node + cây rộng + cây sâu + cây ngẫu nhiên có `seed`?